In [ ]:
import requests

# Test coordinates for Sunamganj, Bangladesh
lat, lon = 25.07, 91.40
test_url = f"https://power.larc.nasa.gov/api/temporal/daily/point?parameters=PRECTOTCORR,T2M&community=AG&longitude={lon}&latitude={lat}&start=20240401&end=20240401&format=JSON"

response = requests.get(test_url)

if response.status_code == 200:
    data = response.json()
    rain = data['properties']['parameter']['PRECTOTCORR']['20240401']
    temp = data['properties']['parameter']['T2M']['20240401']
    print(f"✅ NASA POWER SUCCESS!")
    print(f"Rainfall: {rain}mm | Temp: {temp}°C")
else:
    print(f"❌ NASA POWER FAILED. Error Code: {response.status_code}")

✅ NASA POWER SUCCESS!
Rainfall: 56.57mm | Temp: 25.3°C


In [ ]:
import requests

# Test coordinates for Sunamganj, Bangladesh
lat, lon = 25.07, 91.40
test_url = f"https://power.larc.nasa.gov/api/temporal/daily/point?parameters=PRECTOTCORR,T2M&community=AG&longitude={lon}&latitude={lat}&start=20240401&end=20240401&format=JSON"

response = requests.get(test_url)

if response.status_code == 200:
    data = response.json()
    rain = data['properties']['parameter']['PRECTOTCORR']['20240401']
    temp = data['properties']['parameter']['T2M']['20240401']
    print(f"✅ NASA POWER SUCCESS!")
    print(f"Rainfall: {rain}mm | Temp: {temp}°C")
else:
    print(f"❌ NASA POWER FAILED. Error Code: {response.status_code}")

✅ NASA POWER SUCCESS!
Rainfall: 56.57mm | Temp: 25.3°C


In [ ]:
import ee
ee.Authenticate()

In [ ]:
# 2. Now initialize with your Project ID
ee.Initialize(project='my-ai-agent-481120')

In [ ]:
# 3. Test the connection
test_image = ee.ImageCollection("COPERNICUS/S2_SR").first()
info = test_image.getInfo()

print("✅ GOOGLE EARTH ENGINE SUCCESS!")
print(f"Connected to Image ID: {info['id']}")

✅ GOOGLE EARTH ENGINE SUCCESS!
Connected to Image ID: COPERNICUS/S2_SR/20150704T101006_20150704T101337_T31RFL


/usr/local/lib/python3.12/dist-packages/ee/deprecation.py:207: DeprecationWarning: 

Attention required for COPERNICUS/S2_SR! You are using a deprecated asset.
To make sure your code keeps working, please update it.
Learn more: https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR

  warnings.warn(warning, category=DeprecationWarning)


In [ ]:
import ee

ee.Initialize(project='my-ai-agent-481120')

def get_agro_indices_safe(lat, lon, start_date):
    # 1. Define location
    poi = ee.Geometry.Point([lon, lat])

    # 2. Search for images (we expand the window to 15 days to ensure we find a clear one)
    collection = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
        .filterBounds(poi) \
        .filterDate(start_date, ee.Date(start_date).advance(15, 'day')) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30)) # Slightly higher cloud tolerance

    # 🚨 SAFETY CHECK: Check if the collection is empty
    count = collection.size().getInfo()
    if count == 0:
        return {"error": "No clear satellite images found in this date range."}

    # 3. If images exist, take the best one (Median)
    image = collection.median()

    # 4. Calculate Indices (NDVI and NDWI)
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')

    # 5. Extract results
    combined = image.addBands([ndvi, ndwi])
    stats = combined.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=poi,
        scale=10
    ).getInfo()

    return stats

# Test again for Sunamganj (Northeastern Bangladesh)
result = get_agro_indices_safe(25.07, 91.40, '2024-04-01')
print(result)

{'AOT': 905, 'B1': 757, 'B11': 1910, 'B12': 1234, 'B2': 725, 'B3': 858, 'B4': 716, 'B5': 1259, 'B6': 2097, 'B7': 2423, 'B8': 2558, 'B8A': 2617, 'B9': 2301, 'MSK_CLASSI_CIRRUS': 0, 'MSK_CLASSI_OPAQUE': 0, 'MSK_CLASSI_SNOW_ICE': 0, 'MSK_CLDPRB': 0, 'MSK_SNWPRB': 0, 'NDVI': 0.5626145387904704, 'NDWI': -0.49765807962529274, 'QA10': None, 'QA20': None, 'QA60': 0, 'SCL': 5, 'TCI_B': 74, 'TCI_G': 89, 'TCI_R': 74, 'WVP': 3648}


In [ ]:
import ee
import pandas as pd
from datetime import datetime

# Initialize Earth Engine
#ee.Initialize(project='my-ai-agent-481120')

# 1. Define Location: Sunamganj / Surma River
lat, lon = 25.0714, 91.3992
poi = ee.Geometry.Point([lon, lat])
buffer_zone = poi.buffer(1000) # 1km buffer for river-side accuracy

def get_monthly_data(year, month):
    start_date = f"{year}-{month:02d}-01"
    # Calculate end date (first day of next month)
    if month == 12:
        end_date = f"{year+1}-01-01"
    else:
        end_date = f"{year}-{month+1:02d}-01"

    # --- Sentinel-2 (Crops & Inundation) ---
    s2_col = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
        .filterBounds(buffer_zone) \
        .filterDate(start_date, end_date) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))

    # --- JRC Monthly Water History ---
    jrc_col = ee.ImageCollection("JRC/GSW1_4/MonthlyHistory") \
        .filterBounds(buffer_zone) \
        .filterDate(start_date, end_date)

    row = {'Date': start_date, 'NDVI': None, 'NDWI': None, 'JRC_Water_Fraction': None}

    # Process Sentinel-2
    if s2_col.size().getInfo() > 0:
        img = s2_col.median()
        ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
        ndwi = img.normalizedDifference(['B3', 'B8']).rename('NDWI')

        stats = ndvi.addBands(ndwi).reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=buffer_zone,
            scale=10
        ).getInfo()

        row['NDVI'] = stats.get('NDVI')
        row['NDWI'] = stats.get('NDWI')

    # Process JRC (Water History)
    if jrc_col.size().getInfo() > 0:
        jrc_img = jrc_col.first()
        # water class = 2. Create mask where water exists.
        water_mask = jrc_img.eq(2)

        jrc_stats = water_mask.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=buffer_zone,
            scale=30
        ).getInfo()

        row['JRC_Water_Fraction'] = jrc_stats.get('water')

    return row

# 2. Run Loop for 2017 to 2021
data_results = []
print("Starting Data Extraction for Sunamganj (2017-2021)...")

for year in range(2016, 2022):
    for month in range(1, 13):
        print(f"Processing: {year}-{month:02d}")
        data_results.append(get_monthly_data(year, month))

# 3. Save to CSV
df = pd.DataFrame(data_results)
df.to_csv('sunamganj_surma_history_2017_2021.csv', index=False)
print("Success! File saved as: sunamganj_surma_history_2017_2021.csv")

Starting Data Extraction for Sunamganj (2017-2021)...
Processing: 2016-01
Processing: 2016-02
Processing: 2016-03
Processing: 2016-04
Processing: 2016-05
Processing: 2016-06
Processing: 2016-07
Processing: 2016-08
Processing: 2016-09
Processing: 2016-10
Processing: 2016-11
Processing: 2016-12
Processing: 2017-01
Processing: 2017-02
Processing: 2017-03
Processing: 2017-04
Processing: 2017-05
Processing: 2017-06
Processing: 2017-07
Processing: 2017-08
Processing: 2017-09
Processing: 2017-10
Processing: 2017-11
Processing: 2017-12
Processing: 2018-01
Processing: 2018-02
Processing: 2018-03
Processing: 2018-04
Processing: 2018-05
Processing: 2018-06
Processing: 2018-07
Processing: 2018-08
Processing: 2018-09
Processing: 2018-10
Processing: 2018-11
Processing: 2018-12
Processing: 2019-01
Processing: 2019-02
Processing: 2019-03
Processing: 2019-04
Processing: 2019-05
Processing: 2019-06
Processing: 2019-07
Processing: 2019-08
Processing: 2019-09
Processing: 2019-10
Processing: 2019-11
Proces

**Dahiti api for water level surma river**

In [ ]:
import requests
import json
import pandas as pd
import pprint
from google.colab import files

# --- CONFIGURATION ---
url = "https://dahiti.dgfi.tum.de/api/v2/download-water-level/"

args = {
    'api_key': '362D57A0E203292F23FFE4524D8FC91632BF76AF9C7D9EE91C68EF7FF7F73CAA',
    'dahiti_id': 11199,                   # Surma River, Bangladesh
    'format': 'json',
    'action': 'download-water-level'      # Explicitly state the action
}

# --- EXECUTION ---
print("Requesting data from DAHITI...")
response = requests.post(url, json=args)

if response.status_code == 200:
    data = response.json()

    # Check if the data is wrapped in a list or a dictionary
    water_data = None
    if isinstance(data, list):
        water_data = data
    elif isinstance(data, dict) and 'water_level' in data:
        water_data = data['water_level']

    if water_data and len(water_data) > 0:
        df = pd.DataFrame(water_data)

        # Ensure column names match (DAHITI sometimes uses 'date' or 'datetime')
        date_col = 'date' if 'date' in df.columns else 'datetime'
        df[date_col] = pd.to_datetime(df[date_col])

        # Filter for your range: 2010 to 2025
        df = df[(df[date_col] >= '2010-01-01') & (df[date_col] <= '2025-12-31')]

        # Save and Download
        file_name = 'surma_river_water_levels_2010_2025.csv'
        df.to_csv(file_name, index=False)
        files.download(file_name)

        print(f"✅ Success! {len(df)} points saved to {file_name}")
        print(df.head())
    else:
        print("⚠️ Server returned 200 OK, but the data list is empty.")
        print("Possible Reasons:")
        print("1. Your API key might need a 24-hour activation period.")
        print("2. Station 11199 might be temporarily undergoing maintenance.")
        pprint.pprint(data)
else:
    print(f"❌ HTTP Error {response.status_code}: {response.text}")

Requesting data from DAHITI...
⚠️ Server returned 200 OK, but the data list is empty.
Possible Reasons:
1. Your API key might need a 24-hour activation period.
2. Station 11199 might be temporarily undergoing maintenance.
{'continent': 'Asia',
 'country': 'Bangladesh',
 'creation_date': '2025-12-30 21:27:43',
 'dahiti_id': 11199,
 'data': [{'datetime': '2016-03-12 04:09:52', 'wse': 19.844, 'wse_u': 0.014},
          {'datetime': '2016-04-08 04:09:51', 'wse': 5.634, 'wse_u': 0.044},
          {'datetime': '2016-05-05 04:09:52', 'wse': 6.421, 'wse_u': 0.003},
          {'datetime': '2016-06-01 04:09:52', 'wse': 6.923, 'wse_u': 0.005},
          {'datetime': '2016-06-28 04:09:54', 'wse': 7.204, 'wse_u': 0.0},
          {'datetime': '2016-07-25 04:09:53', 'wse': 9.14, 'wse_u': 0.003},
          {'datetime': '2016-08-21 04:09:52', 'wse': 6.697, 'wse_u': 0.001},
          {'datetime': '2016-09-17 04:09:55', 'wse': 6.297, 'wse_u': 0.012},
          {'datetime': '2016-10-14 04:09:57', 'wse': 6

In [ ]:
data=[{'datetime': '2016-03-12 04:09:52', 'wse': 19.844, 'wse_u': 0.014},
          {'datetime': '2016-04-08 04:09:51', 'wse': 5.634, 'wse_u': 0.044},
          {'datetime': '2016-05-05 04:09:52', 'wse': 6.421, 'wse_u': 0.003},
          {'datetime': '2016-06-01 04:09:52', 'wse': 6.923, 'wse_u': 0.005},
          {'datetime': '2016-06-28 04:09:54', 'wse': 7.204, 'wse_u': 0.0},
          {'datetime': '2016-07-25 04:09:53', 'wse': 9.14, 'wse_u': 0.003},
          {'datetime': '2016-08-21 04:09:52', 'wse': 6.697, 'wse_u': 0.001},
          {'datetime': '2016-09-17 04:09:55', 'wse': 6.297, 'wse_u': 0.012},
          {'datetime': '2016-10-14 04:09:57', 'wse': 6.156, 'wse_u': 0.045},
          {'datetime': '2016-11-10 04:09:53', 'wse': 6.475, 'wse_u': 0.013},
          {'datetime': '2016-12-07 04:09:51', 'wse': 6.184, 'wse_u': 0.075},
          {'datetime': '2017-01-03 04:09:53', 'wse': 6.712, 'wse_u': 0.007},
          {'datetime': '2017-01-30 04:09:54', 'wse': 6.35, 'wse_u': 0.004},
          {'datetime': '2017-02-26 04:09:53', 'wse': 6.497, 'wse_u': 0.01},
          {'datetime': '2017-04-21 04:09:58', 'wse': 6.381, 'wse_u': 0.069},
          {'datetime': '2017-06-14 04:10:03', 'wse': 6.928, 'wse_u': 0.002},
          {'datetime': '2017-07-11 04:10:00', 'wse': 8.022, 'wse_u': 0.004},
          {'datetime': '2017-08-07 04:09:59', 'wse': 7.158, 'wse_u': 0.001},
          {'datetime': '2017-09-03 04:09:54', 'wse': 7.663, 'wse_u': 0.005},
          {'datetime': '2017-09-30 04:09:57', 'wse': 6.696, 'wse_u': 0.006},
          {'datetime': '2017-10-27 04:10:00', 'wse': 6.311, 'wse_u': 0.003},
          {'datetime': '2017-11-23 04:09:58', 'wse': 7.611, 'wse_u': 0.027},
          {'datetime': '2017-12-20 04:09:57', 'wse': 6.498, 'wse_u': 0.015},
          {'datetime': '2018-01-16 04:10:01', 'wse': 6.14, 'wse_u': 0.042},
          {'datetime': '2018-02-12 04:10:02', 'wse': 5.784, 'wse_u': 0.005},
          {'datetime': '2018-05-04 04:10:04', 'wse': 6.844, 'wse_u': 0.036},
          {'datetime': '2018-05-31 04:10:04', 'wse': 6.497, 'wse_u': 0.008},
          {'datetime': '2018-06-27 04:10:05', 'wse': 8.076, 'wse_u': 0.001},
          {'datetime': '2018-07-24 04:10:02', 'wse': 6.824, 'wse_u': 0.045},
          {'datetime': '2018-08-20 04:09:59', 'wse': 6.497, 'wse_u': 0.015},
          {'datetime': '2018-09-16 04:10:02', 'wse': 7.293, 'wse_u': 0.031},
          {'datetime': '2018-10-13 04:10:05', 'wse': 5.924, 'wse_u': 0.006},
          {'datetime': '2018-11-09 04:10:04', 'wse': 6.149, 'wse_u': 0.045},
          {'datetime': '2018-12-06 04:10:01', 'wse': 6.052, 'wse_u': 0.012},
          {'datetime': '2019-01-02 04:10:01', 'wse': 6.276, 'wse_u': 0.009},
          {'datetime': '2019-01-29 04:10:03', 'wse': 6.595, 'wse_u': 0.027},
          {'datetime': '2019-02-25 04:10:03', 'wse': 7.026, 'wse_u': 0.044},
          {'datetime': '2019-03-24 04:10:03', 'wse': 5.951, 'wse_u': 0.052},
          {'datetime': '2019-04-20 04:10:08', 'wse': 7.552, 'wse_u': 0.067},
          {'datetime': '2019-05-17 04:10:10', 'wse': 7.509, 'wse_u': 0.004},
          {'datetime': '2019-06-13 04:10:08', 'wse': 6.589, 'wse_u': 0.01},
          {'datetime': '2019-07-10 04:10:08', 'wse': 8.333, 'wse_u': 0.01},
          {'datetime': '2019-08-06 04:10:06', 'wse': 7.556, 'wse_u': 0.001},
          {'datetime': '2019-09-02 04:10:03', 'wse': 6.632, 'wse_u': 0.032},
          {'datetime': '2019-09-29 04:10:07', 'wse': 6.063, 'wse_u': 0.006},
          {'datetime': '2019-10-26 04:10:07', 'wse': 5.895, 'wse_u': 0.042},
          {'datetime': '2019-11-22 04:10:04', 'wse': 6.205, 'wse_u': 0.048},
          {'datetime': '2019-12-19 04:10:02', 'wse': 6.673, 'wse_u': 0.004},
          {'datetime': '2020-01-15 04:10:05', 'wse': 6.347, 'wse_u': 0.005},
          {'datetime': '2020-02-11 04:10:06', 'wse': 6.455, 'wse_u': 0.051},
          {'datetime': '2020-04-05 04:10:09', 'wse': 4.662, 'wse_u': 0.027},
          {'datetime': '2020-05-02 04:10:12', 'wse': 7.271, 'wse_u': 0.002},
          {'datetime': '2020-05-29 04:10:12', 'wse': 7.753, 'wse_u': 0.013},
          {'datetime': '2020-06-25 04:10:11', 'wse': 8.06, 'wse_u': 0.006},
          {'datetime': '2020-07-22 04:10:11', 'wse': 9.041, 'wse_u': 0.001},
          {'datetime': '2020-08-18 04:10:09', 'wse': 7.44, 'wse_u': 0.001},
          {'datetime': '2020-09-14 04:10:09', 'wse': 7.008, 'wse_u': 0.013},
          {'datetime': '2020-10-11 04:10:11', 'wse': 6.938, 'wse_u': 0.045},
          {'datetime': '2020-11-07 04:10:10', 'wse': 6.291, 'wse_u': 0.037},
          {'datetime': '2020-12-04 04:10:05', 'wse': 6.07, 'wse_u': 0.023},
          {'datetime': '2020-12-31 04:10:08', 'wse': 6.112, 'wse_u': 0.001},
          {'datetime': '2021-01-27 04:10:12', 'wse': 5.603, 'wse_u': 0.021},
          {'datetime': '2021-05-15 04:10:17', 'wse': 6.279, 'wse_u': 0.051},
          {'datetime': '2021-06-11 04:10:17', 'wse': 7.125, 'wse_u': 0.04},
          {'datetime': '2021-07-08 04:10:18', 'wse': 8.314, 'wse_u': 0.006},
          {'datetime': '2021-08-04 04:10:15', 'wse': 6.838, 'wse_u': 0.0},
          {'datetime': '2021-08-31 04:10:13', 'wse': 7.826, 'wse_u': 0.0},
          {'datetime': '2021-10-24 04:10:15', 'wse': 6.874, 'wse_u': 0.0},
          {'datetime': '2021-11-20 04:10:14', 'wse': 6.124, 'wse_u': 0.008},
          {'datetime': '2021-12-17 04:10:12', 'wse': 6.411, 'wse_u': 0.002},
          {'datetime': '2022-01-13 04:10:16', 'wse': 6.382, 'wse_u': 0.016},
          {'datetime': '2022-02-09 04:10:18', 'wse': 5.929, 'wse_u': 0.013},
          {'datetime': '2022-03-08 04:10:15', 'wse': 5.895, 'wse_u': 0.003},
          {'datetime': '2022-04-04 04:10:18', 'wse': 6.043, 'wse_u': 0.011},
          {'datetime': '2022-05-01 04:10:20', 'wse': 6.104, 'wse_u': 0.019},
          {'datetime': '2022-05-28 04:10:18', 'wse': 7.94, 'wse_u': 0.005},
          {'datetime': '2022-06-24 04:10:17', 'wse': 8.669, 'wse_u': 0.013},
          {'datetime': '2022-07-21 04:10:13', 'wse': 7.375, 'wse_u': 0.012},
          {'datetime': '2022-08-17 04:10:11', 'wse': 6.236, 'wse_u': 0.01},
          {'datetime': '2022-09-13 04:10:14', 'wse': 6.387, 'wse_u': 0.032},
          {'datetime': '2022-10-10 04:10:13', 'wse': 6.543, 'wse_u': 0.017},
          {'datetime': '2022-11-06 04:10:12', 'wse': 6.282, 'wse_u': 0.043},
          {'datetime': '2022-12-03 04:10:09', 'wse': 6.033, 'wse_u': 0.088},
          {'datetime': '2022-12-30 04:10:09', 'wse': 6.962, 'wse_u': 0.093},
          {'datetime': '2023-01-26 04:10:13', 'wse': 6.057, 'wse_u': 0.013},
          {'datetime': '2023-03-21 04:10:14', 'wse': 6.068, 'wse_u': 0.011},
          {'datetime': '2023-04-17 04:10:17', 'wse': 5.394, 'wse_u': 0.026},
          {'datetime': '2023-05-14 04:10:15', 'wse': 6.102, 'wse_u': 0.025},
          {'datetime': '2023-06-10 04:10:13', 'wse': 6.954, 'wse_u': 0.02},
          {'datetime': '2023-07-07 04:10:10', 'wse': 8.341, 'wse_u': 0.012},
          {'datetime': '2023-08-03 04:10:07', 'wse': 6.265, 'wse_u': 0.007},
          {'datetime': '2023-08-30 04:10:06', 'wse': 7.751, 'wse_u': 0.001},
          {'datetime': '2023-09-26 04:10:07', 'wse': 7.27, 'wse_u': 0.072},
          {'datetime': '2023-11-19 04:10:07', 'wse': 5.579, 'wse_u': 0.005},
          {'datetime': '2023-12-16 04:10:09', 'wse': 8.337, 'wse_u': 0.083},
          {'datetime': '2024-01-12 04:10:12', 'wse': 5.993, 'wse_u': 0.01},
          {'datetime': '2024-02-08 04:10:12', 'wse': 6.055, 'wse_u': 0.039},
          {'datetime': '2024-04-02 04:10:13', 'wse': 6.976, 'wse_u': 0.091},
          {'datetime': '2024-04-29 04:10:16', 'wse': 6.653, 'wse_u': 0.004},
          {'datetime': '2024-05-26 04:10:15', 'wse': 6.431, 'wse_u': 0.014},
          {'datetime': '2024-06-22 04:10:16', 'wse': 8.732, 'wse_u': 0.002},
          {'datetime': '2024-07-19 04:10:14', 'wse': 7.745, 'wse_u': 0.001},
          {'datetime': '2024-08-15 04:10:10', 'wse': 7.545, 'wse_u': 0.03},
          {'datetime': '2024-09-11 04:10:08', 'wse': 6.427, 'wse_u': 0.001},
          {'datetime': '2024-10-08 04:10:12', 'wse': 6.598, 'wse_u': 0.003},
          {'datetime': '2024-11-04 04:10:12', 'wse': 7.502, 'wse_u': 0.043},
          {'datetime': '2024-12-28 04:10:10', 'wse': 6.503, 'wse_u': 0.014},
          {'datetime': '2025-01-24 04:10:15', 'wse': 5.958, 'wse_u': 0.019},
          {'datetime': '2025-02-20 04:10:14', 'wse': 5.843, 'wse_u': 0.058},
          {'datetime': '2025-03-19 04:10:13', 'wse': 5.304, 'wse_u': 0.088},
          {'datetime': '2025-05-12 04:10:20', 'wse': 6.186, 'wse_u': 0.087},
          {'datetime': '2025-06-08 04:10:20', 'wse': 7.764, 'wse_u': 0.005},
          {'datetime': '2025-07-05 04:10:21', 'wse': 7.309, 'wse_u': 0.021},
          {'datetime': '2025-08-01 04:10:20', 'wse': 7.403, 'wse_u': 0.002},
          {'datetime': '2025-08-28 04:10:17', 'wse': 6.917, 'wse_u': 0.002},
          {'datetime': '2025-09-24 04:10:19', 'wse': 6.817, 'wse_u': 0.039},
          {'datetime': '2025-10-21 04:10:20', 'wse': 6.709, 'wse_u': 0.072},
          {'datetime': '2025-11-17 04:10:19', 'wse': 6.082, 'wse_u': 0.03},
          {'datetime': '2025-12-14 04:10:16', 'wse': 6.449, 'wse_u': 0.018}]
# 2. Create the DataFrame
df = pd.DataFrame(data)

# 3. Clean and Sort
df['datetime'] = pd.to_datetime(df['datetime'])
df = df.sort_values('datetime')

# 4. Save to CSV
csv_name = "surma_river_altimetry_2010_2025.csv"
df.to_csv(csv_name, index=False)

# 5. Trigger the Browser Download
files.download(csv_name)

print(f"✅ CSV file '{csv_name}' has been created and download triggered.")
print(df)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ CSV file 'surma_river_altimetry_2010_2025.csv' has been created and download triggered.
               datetime     wse  wse_u
0   2016-03-12 04:09:52  19.844  0.014
1   2016-04-08 04:09:51   5.634  0.044
2   2016-05-05 04:09:52   6.421  0.003
3   2016-06-01 04:09:52   6.923  0.005
4   2016-06-28 04:09:54   7.204  0.000
..                  ...     ...    ...
114 2025-08-28 04:10:17   6.917  0.002
115 2025-09-24 04:10:19   6.817  0.039
116 2025-10-21 04:10:20   6.709  0.072
117 2025-11-17 04:10:19   6.082  0.030
118 2025-12-14 04:10:16   6.449  0.018

[119 rows x 3 columns]


In [ ]:
import requests
import json
import pandas as pd
import pprint
from google.colab import files

# --- CONFIGURATION ---
url = "https://dahiti.dgfi.tum.de/api/v2/download-water-level/"

args = {
    'api_key': '362D57A0E203292F23FFE4524D8FC91632BF76AF9C7D9EE91C68EF7FF7F73CAA',
    'dahiti_id': 11199,                   # Surma River, Bangladesh
    'format': 'json',
    'action': 'download-water-level'      # Explicitly state the action
}

# --- EXECUTION ---
print("Requesting data from DAHITI...")
response = requests.post(url, json=args)

if response.status_code == 200:
    data = response.json()

    # Check if the data is wrapped in a list or a dictionary
    water_data = None
    if isinstance(data, list):
        water_data = data
    elif isinstance(data, dict) and 'water_level' in data:
        water_data = data['water_level']

    if water_data and len(water_data) > 0:
        df = pd.DataFrame(water_data)

        # Ensure column names match (DAHITI uses 'date' or 'datetime')
        date_col = 'date' if 'date' in df.columns else 'datetime'
        df[date_col] = pd.to_datetime(df[date_col])

        # --- UPDATED FILTER: 2015 to 2016 ---
        start_date = '2015-01-01'
        end_date = '2016-12-31'
        df_filtered = df[(df[date_col] >= start_date) & (df[date_col] <= end_date)]

        if not df_filtered.empty:
            # Save and Download
            file_name = 'surma_river_water_levels_2015_2016.csv'
            df_filtered.to_csv(file_name, index=False)
            files.download(file_name)

            print(f"✅ Success! {len(df_filtered)} points saved to {file_name}")
            print(df_filtered.head())
        else:
            print(f"⚠️ No data points found between {start_date} and {end_date}.")
            print(f"The full dataset range is from {df[date_col].min()} to {df[date_col].max()}.")
    else:
        print("⚠️ Server returned 200 OK, but the data list is empty.")
        pprint.pprint(data)
else:
    print(f"❌ HTTP Error {response.status_code}: {response.text}")


Requesting data from DAHITI...
⚠️ Server returned 200 OK, but the data list is empty.
{'continent': 'Asia',
 'country': 'Bangladesh',
 'creation_date': '2026-01-19 17:03:58',
 'dahiti_id': 11199,
 'data': [{'datetime': '2016-03-12 04:09:52', 'wse': 19.844, 'wse_u': 0.014},
          {'datetime': '2016-04-08 04:09:51', 'wse': 5.634, 'wse_u': 0.044},
          {'datetime': '2016-05-05 04:09:52', 'wse': 6.421, 'wse_u': 0.003},
          {'datetime': '2016-06-01 04:09:52', 'wse': 6.923, 'wse_u': 0.005},
          {'datetime': '2016-06-28 04:09:54', 'wse': 7.204, 'wse_u': 0.0},
          {'datetime': '2016-07-25 04:09:53', 'wse': 9.14, 'wse_u': 0.003},
          {'datetime': '2016-08-21 04:09:52', 'wse': 6.697, 'wse_u': 0.001},
          {'datetime': '2016-09-17 04:09:55', 'wse': 6.297, 'wse_u': 0.012},
          {'datetime': '2016-10-14 04:09:57', 'wse': 6.156, 'wse_u': 0.045},
          {'datetime': '2016-11-10 04:09:53', 'wse': 6.475, 'wse_u': 0.013},
          {'datetime': '2016-12-07 0

In [ ]:
import requests
import pandas as pd
from google.colab import files

# --- 1. SETTINGS ---
lat, lon = 25.07, 91.40
start_date = "20160101"
end_date = "20251231" # Note: NASA data usually has a few days lag
parameters = "PRECTOTCORR,T2M"

# NASA POWER API URL for CSV format (easier for long periods)
url = (f"https://power.larc.nasa.gov/api/temporal/daily/point?"
       f"parameters={parameters}&community=AG&longitude={lon}&latitude={lat}"
       f"&start={start_date}&end={end_date}&format=CSV")

# --- 2. EXECUTION ---
print(f"Requesting NASA POWER data from {start_date} to {end_date}...")
response = requests.get(url)

if response.status_code == 200:
    # Save the text response directly to a CSV file
    file_name = "sunamganj_weather_2016_2025.csv"

    # We need to skip the metadata rows (NASA CSVs have about 10-15 header rows)
    # The actual data starts after the '-END HEADER-' line
    lines = response.text.split('\n')
    header_index = 0
    for i, line in enumerate(lines):
        if "-END HEADER-" in line:
            header_index = i + 1
            break

    # Reconstruct the CSV with only data
    clean_csv_content = '\n'.join(lines[header_index:])

    with open(file_name, "w") as f:
        f.write(clean_csv_content)

    print(f"✅ SUCCESS! Weather data saved to {file_name}")
    files.download(file_name)

    # Quick Check
    df = pd.read_csv(file_name)
    print("\nData Preview:")
    print(df.head())
else:
    print(f"❌ FAILED. Error Code: {response.status_code}")

Requesting NASA POWER data from 20160101 to 20251231...
✅ SUCCESS! Weather data saved to sunamganj_weather_2016_2025.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Data Preview:
   YEAR  DOY  PRECTOTCORR    T2M
0  2016    1          0.0  19.02
1  2016    2          0.0  19.43
2  2016    3          0.0  19.08
3  2016    4          0.0  18.88
4  2016    5          0.0  18.59


In [ ]:
import ee
import pandas as pd
from google.colab import files

# --- 1. INITIALIZE ---
try:
    ee.Initialize(project='my-ai-agent-481120')
except:
    ee.Authenticate()
    ee.Initialize(project='my-ai-agent-481120')

def get_full_feature_set(lat, lon, start_date):
    poi = ee.Geometry.Point([lon, lat])
    end_date = ee.Date(start_date).advance(1, 'month')

    # Filtering for the best available imagery in the month
    collection = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
        .filterBounds(poi) \
        .filterDate(start_date, end_date) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))

    if collection.size().getInfo() == 0:
        return None

    # Median composite to handle temporary clouds/shadows
    image = collection.median()

    # Pre-calculate our primary indices
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')

    # Combine all raw bands + pre-calculated indices
    combined = image.addBands([ndvi, ndwi])

    # Extract mean values of all bands at the POI
    stats = combined.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=poi,
        scale=10
    ).getInfo()

    return stats

# --- 2. EXECUTION LOOP ---
lat, lon = 25.07, 91.40
full_data_list = []

print("🛰️ Extracting Full Feature Set (All Bands + Indices) 2016-2025...")

for year in range(2016, 2026):
    for month in range(1, 13):
        date_str = f"{year}-{month:02d}-01"

        if pd.to_datetime(date_str) > pd.to_datetime('today'):
            break

        print(f"Syncing: {date_str}")
        res = get_full_feature_set(lat, lon, date_str)

        if res:
            res['Date'] = date_str  # Ensure date is a column
            full_data_list.append(res)

# --- 3. EXPORT TO CSV ---
if full_data_list:
    df_full = pd.DataFrame(full_data_list)

    # Move 'Date' to the first column for better organization
    cols = ['Date'] + [c for c in df_full.columns if c != 'Date']
    df_full = df_full[cols]

    output_name = "sunamganj_sentinel_full_features_2016_2025.csv"
    df_full.to_csv(output_name, index=False)

    print(f"\n✅ DOWNLOAD READY: {len(df_full)} samples with {len(df_full.columns)} features.")
    files.download(output_name)
else:
    print("❌ No data found.")

🛰️ Extracting Full Feature Set (All Bands + Indices) 2016-2025...
Syncing: 2016-01-01
Syncing: 2016-02-01
Syncing: 2016-03-01
Syncing: 2016-04-01
Syncing: 2016-05-01
Syncing: 2016-06-01
Syncing: 2016-07-01
Syncing: 2016-08-01
Syncing: 2016-09-01
Syncing: 2016-10-01
Syncing: 2016-11-01
Syncing: 2016-12-01
Syncing: 2017-01-01
Syncing: 2017-02-01
Syncing: 2017-03-01
Syncing: 2017-04-01
Syncing: 2017-05-01
Syncing: 2017-06-01
Syncing: 2017-07-01
Syncing: 2017-08-01
Syncing: 2017-09-01
Syncing: 2017-10-01
Syncing: 2017-11-01
Syncing: 2017-12-01
Syncing: 2018-01-01
Syncing: 2018-02-01
Syncing: 2018-03-01
Syncing: 2018-04-01
Syncing: 2018-05-01
Syncing: 2018-06-01
Syncing: 2018-07-01
Syncing: 2018-08-01
Syncing: 2018-09-01
Syncing: 2018-10-01
Syncing: 2018-11-01
Syncing: 2018-12-01
Syncing: 2019-01-01
Syncing: 2019-02-01
Syncing: 2019-03-01
Syncing: 2019-04-01
Syncing: 2019-05-01
Syncing: 2019-06-01
Syncing: 2019-07-01
Syncing: 2019-08-01
Syncing: 2019-09-01
Syncing: 2019-10-01
Syncing: 2019-

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# 1. Install and Import Libraries
!pip install geemap
import ee
import geemap

# 2. Authenticate and Initialize Earth Engine
try:
    ee.Initialize(project='my-ai-agent-481120')
except:
    ee.Authenticate()
    ee.Initialize(project='my-ai-agent-481120')

# 3. Define Coordinates and Area
lat, lon = 25.0714, 91.3992

# Load Sunamganj District Boundary
district = ee.FeatureCollection("FAO/GAUL/2015/level2") \
            .filter(ee.Filter.eq('ADM2_NAME', 'Sunamganj'))

# 4. Load Sentinel-2 Data (April 2024 - Harvest Season)
image = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(district) \
    .filterDate('2024-04-01', '2024-04-30') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
    .median() \
    .clip(district)

# 5. Load ESA WorldCover and create Crop Mask
# Class 40 is 'Cropland'
landcover = ee.Image("ESA/WorldCover/v100/2020")
crop_mask = landcover.eq(40)

# Apply mask to keep ONLY the rice fields
masked_satellite = image.updateMask(crop_mask)

# 6. Setup Interactive Map
Map = geemap.Map(center=[lat, lon], zoom=10)

# Define Visualization (True Color RGB)
vis_params = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 3000,
    'gamma': 1.4
}

# Add Layers to Map
Map.add_basemap('SATELLITE')
Map.addLayer(masked_satellite, vis_params, 'Sunamganj Masked Crop Fields')
Map.addLayer(district, {'color': 'red'}, 'District Boundary', False)

# 7. Display Map
Map

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 11.0 MB/s eta 0:00:00


Map(center=[25.0714, 91.3992], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDa…

**Sunamgonj visualization**

In [ ]:
# 1. Install and Import Libraries
!pip install geemap
import ee
import geemap

# 2. Authenticate and Initialize Earth Engine
try:
    ee.Initialize(project='my-ai-agent-481120')
except:
    ee.Authenticate()
    ee.Initialize(project='my-ai-agent-481120')

# 3. Define Coordinates for Sunamganj
lat, lon = 25.0714, 91.3992

# Load Sunamganj District Boundary
district = ee.FeatureCollection("FAO/GAUL/2015/level2") \
            .filter(ee.Filter.eq('ADM2_NAME', 'Sunamganj'))

# 4. Load Sentinel-2 Data for JUNE 2025
# We use a 30-day window to find the clearest possible composite
image = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(district) \
    .filterDate('2025-06-01', '2025-06-30') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 40)) \
    .median() \
    .clip(district)

# 5. Load ESA WorldCover and create Crop Mask (Class 40 = Cropland)
landcover = ee.Image("ESA/WorldCover/v100/2020")
crop_mask = landcover.eq(40)

# Mask the satellite image to show ONLY the fields
masked_satellite = image.updateMask(crop_mask)

# 6. Calculate NDVI to see health/submergence
ndvi = masked_satellite.normalizedDifference(['B8', 'B4']).rename('NDVI')

# 7. Setup Interactive Map
Map = geemap.Map(center=[lat, lon], zoom=10)

# True Color Visualization (Natural look)
vis_params = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 3000,
    'gamma': 1.4
}

# NDVI Visualization (Red-Yellow-Green)
ndvi_vis = {
    'min': 0,
    'max': 0.8,
    'palette': ['#e74c3c', '#f1c40f', '#2ecc71'] # Red to Yellow to Green
}

# Add Layers
Map.add_basemap('SATELLITE')
Map.addLayer(masked_satellite, vis_params, 'Sunamganj Crop Fields (June 2025)')
Map.addLayer(ndvi, ndvi_vis, 'NDVI Health (June 2025)')
Map.addLayer(district, {'color': 'red'}, 'District Boundary', False)

# Display Map
Map

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 10.9 MB/s eta 0:00:00


Map(center=[25.0714, 91.3992], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDa…

**Sunamgonj_Harvest_Data_with_MAshing**

In [ ]:
import ee
ee.Authenticate()

True

In [ ]:
import ee
import pandas as pd
import datetime
import numpy as np

# Initialize
try:
    ee.Initialize(project='my-ai-agent-481120')
except:
    ee.Authenticate()
    ee.Initialize(project='my-ai-agent-481120')


def generate_sunamganj_harvest_features_csv():
    print("🚜 Sunamganj Crop Harvest + Feature Extraction...")

    # -------- REGION (Sunamganj Haor) --------
    roi = ee.Geometry.Point([91.3992, 25.0710]).buffer(20000)

    # -------- PREPROCESSING --------
    def prep_l8(image):
        qa = image.select('QA_PIXEL')
        mask = qa.bitwiseAnd(1 << 3).eq(0).And(qa.bitwiseAnd(1 << 4).eq(0))

        img = (image
            .select(['SR_B2','SR_B3','SR_B4','SR_B5'],
                    ['Blue','Green','Red','NIR'])
            .multiply(0.0000275).add(-0.2)
            .updateMask(mask)
            .toFloat())

        return img.set('system:time_start', image.get('system:time_start'))

    def prep_l7(image):
        qa = image.select('QA_PIXEL')
        mask = qa.bitwiseAnd(1 << 3).eq(0).And(qa.bitwiseAnd(1 << 4).eq(0))

        img = (image
            .select(['SR_B1','SR_B2','SR_B3','SR_B4'],
                    ['Blue','Green','Red','NIR'])
            .multiply(0.0000275).add(-0.2)
            .updateMask(mask)
            .toFloat())

        return img.set('system:time_start', image.get('system:time_start'))

    l8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").filterBounds(roi).map(prep_l8)
    l7 = ee.ImageCollection("LANDSAT/LE07/C02/T1_L2").filterBounds(roi).map(prep_l7)

    collection = l8.merge(l7).sort('system:time_start')

    # -------- DATE RANGE --------
    start_date = datetime.date(2020, 1, 1)
    end_date   = datetime.date(2023, 12, 31)

    records = []
    current = start_date

    # -------- Helper: short-term NDVI trend --------
    previous_ndvi = None

    while current <= end_date:
        next_date = current + datetime.timedelta(days=5)
        d1 = current.strftime('%Y-%m-%d')
        d2 = next_date.strftime('%Y-%m-%d')

        try:
            window = collection.filterDate(d1, d2)
            count = window.limit(1).size().getInfo()

            if count > 0:

                img = window.median().clip(roi)

                # ---------- CORE INDICES ----------
                ndvi = img.normalizedDifference(['NIR','Red']).rename('NDVI')
                ndwi = img.normalizedDifference(['Green','NIR']).rename('NDWI')
                lswi = img.normalizedDifference(['NIR','Green']).rename('LSWI')

                # EVI
                evi = img.expression(
                    '2.5*((NIR-Red)/(NIR+6*Red-7.5*Blue+1))',
                    {'NIR': img.select('NIR'),
                     'Red': img.select('Red'),
                     'Blue': img.select('Blue')}
                ).rename('EVI')

                # SAVI (L = 0.5)
                savi = img.expression(
                    '((1+0.5)*(NIR-Red))/(NIR+Red+0.5)',
                    {'NIR': img.select('NIR'),
                     'Red': img.select('Red')}
                ).rename('SAVI')

                # Green Chlorophyll Index
                gci = img.expression(
                    '(NIR/Green) - 1',
                    {'NIR': img.select('NIR'),
                     'Green': img.select('Green')}
                ).rename('GCI')

                # LAI Proxy (empirical NDVI relation)
                lai = ndvi.expression(
                    '-log((0.69-NDVI)/0.59)',
                    {'NDVI': ndvi}
                ).rename('LAI_proxy')

                # Dryness proxy (NMDI-like)
                nmdi = img.expression(
                    '(NIR-Red)/(NIR+Red)',
                    {'NIR': img.select('NIR'),
                     'Red': img.select('Red')}
                ).rename('Dryness_Proxy')

                # ---------- HARVEST CLASS LOGIC ----------
                classification = (ee.Image(0)
                    .where(ndvi.gte(0.65), 1)      # Growing
                    .where(ndvi.gte(0.45)
                           .And(ndvi.lt(0.55)), 2) # Ready to Harvest
                )

                crop_mask = ndvi.gte(0.35)

                hist = (classification.updateMask(crop_mask)
                    .reduceRegion(
                        reducer=ee.Reducer.frequencyHistogram(),
                        geometry=roi,
                        scale=30,
                        maxPixels=1e9,
                        bestEffort=True)
                    .getInfo()
                    .get('constant', {}))

                total = sum(hist.values()) if hist else 1

                # ---------- MEAN FEATURE EXTRACTION ----------
                feature_stack = img.select(['Blue','Green','Red']) \
                    .addBands([ndvi, ndwi, lswi, evi, savi, gci, lai, nmdi])

                means = feature_stack.reduceRegion(
                    reducer=ee.Reducer.mean(),
                    geometry=roi,
                    scale=30,
                    maxPixels=1e9,
                    bestEffort=True
                ).getInfo()

                # ---------- NDVI TREND ----------
                ndvi_value = means.get('NDVI')
                trend = None
                if previous_ndvi is not None and ndvi_value is not None:
                    trend = ndvi_value - previous_ndvi
                previous_ndvi = ndvi_value

                # ---------- RECORD ROW ----------
                records.append({
                    'Date': d1,

                    # Spectral features
                    'Mean_NDVI': means.get('NDVI'),
                    'Mean_EVI': means.get('EVI'),
                    'Mean_SAVI': means.get('SAVI'),
                    'Mean_GCI': means.get('GCI'),
                    'Mean_LAI_Proxy': means.get('LAI_proxy'),

                    # Moisture / senescence
                    'Mean_NDWI': means.get('NDWI'),
                    'Mean_LSWI': means.get('LSWI'),
                    'Mean_Dryness': means.get('Dryness_Proxy'),

                    # Time-series support
                    'NDVI_Trend': trend,

                    # RGB reflectance
                    'Mean_Blue': means.get('Blue'),
                    'Mean_Green': means.get('Green'),
                    'Mean_Red': means.get('Red'),

                    # Harvest status outputs
                    'Pct_Growing': (hist.get('1', 0) / total) * 100,
                    'Pct_Ready_Harvest': (hist.get('2', 0) / total) * 100
                })

                print(f"✓ {d1}  ReadyHarvest={records[-1]['Pct_Ready_Harvest']:.1f}% "
                      f" NDVI={means.get('NDVI'):.2f}")

            else:
                records.append({'Date': d1, 'Mean_NDVI': np.nan})

        except Exception as e:
            print(f"✗ {d1} Error: {e}")
            records.append({'Date': d1, 'Mean_NDVI': np.nan})

        current = next_date

    # -------- SAVE CSV --------
    print("\n💾 Saving CSV with extended crop features...")
    df = pd.DataFrame(records)
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.set_index('Date').sort_index()

    df = df.interpolate(method='time', limit=2)
    df.reset_index(inplace=True)

    filename = 'sunamganj_crop_harvest_advanced_features(2016to2019).csv'
    df.to_csv(filename, index=False)

    print("🎉 Saved:", filename)
    print(df.head())


# Run
generate_sunamganj_harvest_features_csv()


🚜 Sunamganj Crop Harvest + Feature Extraction...
✓ 2016-01-06  ReadyHarvest=31.4%  NDVI=0.32
✓ 2016-01-11  ReadyHarvest=19.3%  NDVI=0.31
✓ 2016-01-16  ReadyHarvest=0.0%  NDVI=0.11
✓ 2016-01-21  ReadyHarvest=27.7%  NDVI=0.33
✓ 2016-01-31  ReadyHarvest=23.1%  NDVI=0.39
✓ 2016-02-05  ReadyHarvest=32.7%  NDVI=0.41
✓ 2016-02-15  ReadyHarvest=34.4%  NDVI=0.44
✓ 2016-02-20  ReadyHarvest=34.3%  NDVI=0.37
✓ 2016-02-25  ReadyHarvest=23.4%  NDVI=0.49
✓ 2016-03-01  ReadyHarvest=29.2%  NDVI=0.51
✓ 2016-03-11  ReadyHarvest=32.2%  NDVI=0.47
✓ 2016-03-16  ReadyHarvest=31.8%  NDVI=0.52
✓ 2016-03-26  ReadyHarvest=4.7%  NDVI=0.24
✓ 2016-03-31  ReadyHarvest=21.1%  NDVI=0.59
✓ 2016-04-05  ReadyHarvest=0.0%  NDVI=-0.47
✓ 2016-04-10  ReadyHarvest=43.8%  NDVI=0.50
✓ 2016-04-20  ReadyHarvest=18.9%  NDVI=0.47
✓ 2016-04-25  ReadyHarvest=33.7%  NDVI=0.21
✓ 2016-05-05  ReadyHarvest=31.0%  NDVI=0.12
✓ 2016-05-10  ReadyHarvest=34.2%  NDVI=0.36
✓ 2016-05-15  ReadyHarvest=100.0%  NDVI=0.26
✓ 2016-05-20  ReadyHarvest=2

*Double_check***

**Advance Feature**

In [ ]:
import ee
import pandas as pd
import datetime
import numpy as np

# 1. Initialize
try:
    ee.Initialize(project='my-ai-agent-481120')
except:
    ee.Authenticate()
    ee.Initialize(project='my-ai-agent-481120')

def generate_sunamganj_cloud_proof_data():
    print(f"🛰️ AgroPulse-BD: Cloud-Proof Radar + Optical Fusion...")

    # -------- DISTRICT BOUNDARY --------
    districts = ee.FeatureCollection("FAO/GAUL/2015/level2")
    roi = districts.filter(ee.Filter.eq('ADM2_NAME', 'Sunamganj')).geometry()
    esa_crop_mask = ee.Image("ESA/WorldCover/v100/2020").select('Map').clip(roi).eq(40)

    # -------- OPTICAL PROCESSING (Landsat + Sentinel-2) --------
    def prep_optical(image):
        # Universal NDVI calculation for merged collections
        # S2 uses B8/B4, Landsat uses B5/B4
        ndvi = image.normalizedDifference(['NIR', 'Red']).rename('NDVI')
        return image.addBands(ndvi).select('NDVI').toFloat()

    s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(roi) \
           .map(lambda img: img.select(['B8','B4'],['NIR','Red']).divide(10000).set('system:time_start', img.get('system:time_start')))
    l8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").filterBounds(roi) \
           .map(lambda img: img.select(['SR_B5','SR_B4'],['NIR','Red']).multiply(0.0000275).add(-0.2).set('system:time_start', img.get('system:time_start')))

    optical_col = s2.merge(l8).map(prep_optical)

    # -------- RADAR PROCESSING (Sentinel-1) --------
    # Radar sees through clouds! We use VV polarization to detect water/crops.
    s1 = ee.ImageCollection("COPERNICUS/S1_GRD") \
           .filterBounds(roi) \
           .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
           .filter(ee.Filter.eq('instrumentMode', 'IW'))

    # -------- ANALYSIS LOOP --------
    start_date, end_date = datetime.date(2024, 1, 1), datetime.date(2024, 12, 31)
    records, current = [], start_date

    while current <= end_date:
        d1 = current.strftime('%Y-%m-%d')
        # Extended 20-day window to maximize the chance of a cloud-free optical gap
        d2 = (current + datetime.timedelta(days=20)).strftime('%Y-%m-%d')

        try:
            # 1. Try Optical first
            opt_window = optical_col.filterDate(d1, d2)
            opt_val = None

            if opt_window.size().getInfo() > 0:
                opt_comp = opt_window.qualityMosaic('NDVI').clip(roi).updateMask(esa_crop_mask)
                opt_stats = opt_comp.reduceRegion(reducer=ee.Reducer.mean(), geometry=roi, scale=250, tileScale=4).getInfo()
                opt_val = opt_stats.get('NDVI')

            # 2. Use Radar (Sentinel-1) as a proxy if Optical is cloudy or low
            # VV backscatter increases as crops grow taller out of the water
            sar_window = s1.filterDate(d1, d2)
            sar_val = None
            if sar_window.size().getInfo() > 0:
                sar_comp = sar_window.median().clip(roi).updateMask(esa_crop_mask)
                sar_stats = sar_comp.reduceRegion(reducer=ee.Reducer.mean(), geometry=roi, scale=250, tileScale=4).getInfo()
                sar_val = sar_stats.get('VV')

            # Determine Data Status
            status = "✅ Success" if opt_val else "📡 Radar Only"

            records.append({
                'Date': d1,
                'Mean_NDVI': opt_val,
                'Radar_VV': sar_val,
                'Status': status
            })
            print(f"{status} {d1} | NDVI: {opt_val if opt_val else 'CLOUDY':.3f} | Radar VV: {sar_val:.2f}")

        except Exception as e:
            records.append({'Date': d1, 'Mean_NDVI': np.nan, 'Status': 'Error'})

        current += datetime.timedelta(days=5)

    # -------- FINAL STEP: INTELLIGENT FILL --------
    df = pd.DataFrame(records)
    # If NDVI is missing, we use linear interpolation to fill the gap
    df['Mean_NDVI'] = df['Mean_NDVI'].interpolate(limit_direction='both')

    df.to_csv('sunamganj_cloudproof_boro_data.csv', index=False)
    print("\n🎉 Process complete. Check 'sunamganj_cloudproof_boro_data.csv'")

if __name__ == "__main__":
    generate_sunamganj_cloud_proof_data()

🛰️ AgroPulse-BD: Cloud-Proof Radar + Optical Fusion...
✅ Success 2024-01-01 | NDVI: 0.355 | Radar VV: -11.06
✅ Success 2024-01-06 | NDVI: 0.339 | Radar VV: -10.76
✅ Success 2024-01-11 | NDVI: 0.395 | Radar VV: -10.60
✅ Success 2024-01-16 | NDVI: 0.446 | Radar VV: -10.30
✅ Success 2024-01-21 | NDVI: 0.467 | Radar VV: -10.10
✅ Success 2024-01-26 | NDVI: 0.542 | Radar VV: -10.03
✅ Success 2024-01-31 | NDVI: 0.557 | Radar VV: -9.74
✅ Success 2024-02-05 | NDVI: 0.556 | Radar VV: -9.83
✅ Success 2024-02-10 | NDVI: 0.552 | Radar VV: -9.68
✅ Success 2024-02-15 | NDVI: 0.668 | Radar VV: -9.85
✅ Success 2024-02-20 | NDVI: 0.696 | Radar VV: -9.96
✅ Success 2024-02-25 | NDVI: 0.698 | Radar VV: -10.31
✅ Success 2024-03-01 | NDVI: 0.707 | Radar VV: -10.51
✅ Success 2024-03-06 | NDVI: 0.711 | Radar VV: -10.62
✅ Success 2024-03-11 | NDVI: 0.694 | Radar VV: -10.90
✅ Success 2024-03-16 | NDVI: 0.696 | Radar VV: -10.69
✅ Success 2024-03-21 | NDVI: 0.630 | Radar VV: -10.28


✅ Success 2024-03-26 | NDVI: 0.567 | Radar VV: -10.14
✅ Success 2024-03-31 | NDVI: 0.526 | Radar VV: -9.23
✅ Success 2024-04-05 | NDVI: 0.516 | Radar VV: -9.72
✅ Success 2024-04-10 | NDVI: 0.532 | Radar VV: -10.06
✅ Success 2024-04-15 | NDVI: 0.532 | Radar VV: -9.64
✅ Success 2024-04-20 | NDVI: 0.493 | Radar VV: -9.60
✅ Success 2024-04-25 | NDVI: 0.494 | Radar VV: -9.01
✅ Success 2024-04-30 | NDVI: 0.453 | Radar VV: -8.95
✅ Success 2024-05-05 | NDVI: 0.470 | Radar VV: -8.69
✅ Success 2024-05-10 | NDVI: 0.484 | Radar VV: -8.64
✅ Success 2024-05-15 | NDVI: 0.454 | Radar VV: -8.66
✅ Success 2024-05-20 | NDVI: 0.453 | Radar VV: -11.16
✅ Success 2024-05-25 | NDVI: 0.408 | Radar VV: -13.68
✅ Success 2024-05-30 | NDVI: 0.039 | Radar VV: -14.45
✅ Success 2024-06-04 | NDVI: 0.037 | Radar VV: -15.98
✅ Success 2024-06-09 | NDVI: 0.036 | Radar VV: -15.92
✅ Success 2024-06-14 | NDVI: 0.024 | Radar VV: -15.38
✅ Success 2024-06-19 | NDVI: 0.023 | Radar VV: -16.63
✅ Success 2024-06-24 | NDVI: 0.002 | 

✅ Success 2024-08-18 | NDVI: 0.197 | Radar VV: -15.15
✅ Success 2024-08-23 | NDVI: 0.194 | Radar VV: -15.23
✅ Success 2024-08-28 | NDVI: 0.184 | Radar VV: -15.49
✅ Success 2024-09-02 | NDVI: 0.227 | Radar VV: -14.94
✅ Success 2024-09-07 | NDVI: 0.247 | Radar VV: -14.94
✅ Success 2024-09-12 | NDVI: 0.277 | Radar VV: -14.29
✅ Success 2024-09-17 | NDVI: 0.288 | Radar VV: -13.74
✅ Success 2024-09-22 | NDVI: 0.257 | Radar VV: -14.18
✅ Success 2024-09-27 | NDVI: 0.269 | Radar VV: -13.85
✅ Success 2024-10-02 | NDVI: 0.322 | Radar VV: -14.77
✅ Success 2024-10-07 | NDVI: 0.349 | Radar VV: -14.86
✅ Success 2024-10-12 | NDVI: 0.372 | Radar VV: -14.61
✅ Success 2024-10-17 | NDVI: 0.373 | Radar VV: -14.43
✅ Success 2024-10-22 | NDVI: 0.389 | Radar VV: -13.73
✅ Success 2024-10-27 | NDVI: 0.397 | Radar VV: -13.70
✅ Success 2024-11-01 | NDVI: 0.410 | Radar VV: -13.22
✅ Success 2024-11-06 | NDVI: 0.397 | Radar VV: -13.15
✅ Success 2024-11-11 | NDVI: 0.381 | Radar VV: -12.61
✅ Success 2024-11-16 | NDVI:

**Sunamgonj NDVI value For crop hervest final.**

In [ ]:
import ee
import pandas as pd
import datetime
import numpy as np

# 1. Initialize Earth Engine
try:
    ee.Initialize(project='my-ai-agent-481120')
except:
    ee.Authenticate()
    ee.Initialize(project='my-ai-agent-481120')

def generate_sunamganj_cloud_proof_data():
    print("🛰️ AgroPulse-BD: Optimized NDVI Pipeline Running...")

    # -------- DISTRICT BOUNDARY --------
    districts = ee.FeatureCollection("FAO/GAUL/2015/level2")
    roi = districts.filter(ee.Filter.eq('ADM2_NAME', 'Sunamganj')).geometry()

    # Cropland mask (ESA WorldCover – Class 40)
    crop_mask = (
        ee.Image("ESA/WorldCover/v100/2020")
        .select('Map')
        .eq(40)
        .clip(roi)
    )

    # -------- OPTICAL PROCESSING --------
    def prep_optical(image):
        ndvi = image.normalizedDifference(['NIR', 'Red']).rename('NDVI')
        return image.addBands(ndvi).select('NDVI').toFloat()

    # Sentinel-2
    s2 = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(roi)
        .map(lambda img: img
             .select(['B8', 'B4'], ['NIR', 'Red'])
             .divide(10000)
             .copyProperties(img, ['system:time_start']))
    )

    # Landsat-8
    l8 = (
        ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
        .filterBounds(roi)
        .map(lambda img: img
             .select(['SR_B5', 'SR_B4'], ['NIR', 'Red'])
             .multiply(0.0000275)
             .add(-0.2)
             .copyProperties(img, ['system:time_start']))
    )

    optical_col = s2.merge(l8).map(prep_optical)

    # -------- RADAR PROCESSING (Sentinel-1) --------
    s1 = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(roi)
        .filter(ee.Filter.eq('instrumentMode', 'IW'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
        .select('VV')
    )

    # -------- ANALYSIS LOOP --------
    start_date = datetime.date(2025, 1, 1)
    end_date   = datetime.date(2025, 12, 31)

    records = []
    current = start_date

    while current <= end_date:
        d1 = current.strftime('%Y-%m-%d')
        d2 = (current + datetime.timedelta(days=20)).strftime('%Y-%m-%d')

        opt_val, sar_val = None, None

        try:
            # Optical NDVI
            opt_window = optical_col.filterDate(d1, d2)
            if opt_window.size().getInfo() > 0:
                opt_comp = (
                    opt_window
                    .qualityMosaic('NDVI')
                    .updateMask(crop_mask)
                    .clip(roi)
                )

                stats = opt_comp.reduceRegion(
                    reducer=ee.Reducer.mean(),
                    geometry=roi,
                    scale=30,
                    tileScale=4,
                    maxPixels=1e13
                ).getInfo()

                opt_val = stats.get('NDVI')

            # Radar VV
            sar_window = s1.filterDate(d1, d2)
            if sar_window.size().getInfo() > 0:
                sar_comp = (
                    sar_window
                    .median()
                    .updateMask(crop_mask)
                    .clip(roi)
                )

                sar_stats = sar_comp.reduceRegion(
                    reducer=ee.Reducer.mean(),
                    geometry=roi,
                    scale=30,
                    tileScale=4,
                    maxPixels=1e13
                ).getInfo()

                sar_val = sar_stats.get('VV')

            status = "✅ Optical NDVI" if opt_val is not None else "📡 Radar Only"

            records.append({
                'Date': d1,
                'Mean_NDVI': opt_val,
                'Radar_VV': sar_val,
                'Status': status
            })

            ndvi_str = f"{opt_val:.3f}" if opt_val is not None else "CLOUDY"
            vv_str   = f"{sar_val:.2f}" if sar_val is not None else "NA"

            print(f"{status} | {d1} | NDVI: {ndvi_str} | VV: {vv_str}")

        except Exception as e:
            records.append({
                'Date': d1,
                'Mean_NDVI': np.nan,
                'Radar_VV': np.nan,
                'Status': 'Error'
            })

        current += datetime.timedelta(days=5)

    # -------- FINAL STEP: FILL NDVI GAPS --------
    df = pd.DataFrame(records)
    df['Mean_NDVI'] = df['Mean_NDVI'].interpolate(limit_direction='both')

    df.to_csv('sunamganj_ndvi_cropland_timeseries(2025).csv', index=False)
    print("\n🎉 NDVI time-series generated successfully!")

if __name__ == "__main__":
    generate_sunamganj_cloud_proof_data()


🛰️ AgroPulse-BD: Optimized NDVI Pipeline Running...
✅ Optical NDVI | 2025-01-01 | NDVI: 0.388 | VV: -11.69
✅ Optical NDVI | 2025-01-06 | NDVI: 0.388 | VV: -11.58
✅ Optical NDVI | 2025-01-11 | NDVI: 0.391 | VV: -11.18
✅ Optical NDVI | 2025-01-16 | NDVI: 0.444 | VV: -10.80
✅ Optical NDVI | 2025-01-21 | NDVI: 0.559 | VV: -10.70
✅ Optical NDVI | 2025-01-26 | NDVI: 0.594 | VV: -10.50
✅ Optical NDVI | 2025-01-31 | NDVI: 0.599 | VV: -10.56
✅ Optical NDVI | 2025-02-05 | NDVI: 0.647 | VV: -10.56
✅ Optical NDVI | 2025-02-10 | NDVI: 0.646 | VV: -10.92
✅ Optical NDVI | 2025-02-15 | NDVI: 0.658 | VV: -11.03
✅ Optical NDVI | 2025-02-20 | NDVI: 0.657 | VV: -11.45
✅ Optical NDVI | 2025-02-25 | NDVI: 0.642 | VV: -11.87
✅ Optical NDVI | 2025-03-02 | NDVI: 0.641 | VV: -11.76
✅ Optical NDVI | 2025-03-07 | NDVI: 0.678 | VV: -12.29
✅ Optical NDVI | 2025-03-12 | NDVI: 0.686 | VV: -12.25
✅ Optical NDVI | 2025-03-17 | NDVI: 0.686 | VV: -11.66
✅ Optical NDVI | 2025-03-22 | NDVI: 0.704 | VV: -11.63
✅ Optical NDV

✅ Optical NDVI | 2025-08-19 | NDVI: 0.311 | VV: -15.89
✅ Optical NDVI | 2025-08-24 | NDVI: 0.305 | VV: -16.08
✅ Optical NDVI | 2025-08-29 | NDVI: 0.304 | VV: -15.74
✅ Optical NDVI | 2025-09-03 | NDVI: 0.336 | VV: -16.17
✅ Optical NDVI | 2025-09-08 | NDVI: 0.340 | VV: -16.06
✅ Optical NDVI | 2025-09-13 | NDVI: 0.355 | VV: -15.26
✅ Optical NDVI | 2025-09-18 | NDVI: 0.357 | VV: -15.12
✅ Optical NDVI | 2025-09-23 | NDVI: 0.352 | VV: -14.71
✅ Optical NDVI | 2025-09-28 | NDVI: 0.402 | VV: -14.88
✅ Optical NDVI | 2025-10-03 | NDVI: 0.429 | VV: -15.01
✅ Optical NDVI | 2025-10-08 | NDVI: 0.407 | VV: -15.31
✅ Optical NDVI | 2025-10-13 | NDVI: 0.444 | VV: -14.96
✅ Optical NDVI | 2025-10-18 | NDVI: 0.446 | VV: -14.72
✅ Optical NDVI | 2025-10-23 | NDVI: 0.443 | VV: -14.60
✅ Optical NDVI | 2025-10-28 | NDVI: 0.448 | VV: -13.97
✅ Optical NDVI | 2025-11-02 | NDVI: 0.408 | VV: -14.15
✅ Optical NDVI | 2025-11-07 | NDVI: 0.427 | VV: -13.83
✅ Optical NDVI | 2025-11-12 | NDVI: 0.392 | VV: -13.62
✅ Optical 

**Sunamgonj NDWI value For crop hervest final.**

In [ ]:
import ee
import pandas as pd
import datetime
import numpy as np

# 1. Initialize Earth Engine
try:
    ee.Initialize(project='my-ai-agent-481120')
except:
    ee.Authenticate()
    ee.Initialize(project='my-ai-agent-481120')

def generate_sunamganj_water_data():
    print("🛰️ AgroPulse-BD: Optimized NDWI & NDVI Pipeline Running...")

    # -------- DISTRICT BOUNDARY --------
    districts = ee.FeatureCollection("FAO/GAUL/2015/level2")
    roi = districts.filter(ee.Filter.eq('ADM2_NAME', 'Sunamganj')).geometry()

    # Cropland mask (ESA WorldCover – Class 40)
    crop_mask = (
        ee.Image("ESA/WorldCover/v100/2020")
        .select('Map')
        .eq(40)
        .clip(roi)
    )

    # -------- OPTICAL PROCESSING --------
    def prep_optical(image):
        # NDVI calculation
        ndvi = image.normalizedDifference(['NIR', 'Red']).rename('NDVI')
        # NDWI calculation (Green - NIR) / (Green + NIR)
        ndwi = image.normalizedDifference(['Green', 'NIR']).rename('NDWI')
        return image.addBands([ndvi, ndwi]).select(['NDVI', 'NDWI']).toFloat()

    # Sentinel-2
    s2 = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(roi)
        .map(lambda img: img
             .select(['B8', 'B4', 'B3'], ['NIR', 'Red', 'Green'])
             .divide(10000)
             .copyProperties(img, ['system:time_start']))
    )

    # Landsat-8
    l8 = (
        ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
        .filterBounds(roi)
        .map(lambda img: img
             .select(['SR_B5', 'SR_B4', 'SR_B3'], ['NIR', 'Red', 'Green'])
             .multiply(0.0000275)
             .add(-0.2)
             .copyProperties(img, ['system:time_start']))
    )

    optical_col = s2.merge(l8).map(prep_optical)

    # -------- RADAR PROCESSING (Sentinel-1) --------
    s1 = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(roi)
        .filter(ee.Filter.eq('instrumentMode', 'IW'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
        .select('VV')
    )

    # -------- ANALYSIS LOOP --------
    start_date = datetime.date(2017, 1, 1)
    end_date   = datetime.date(2022, 12, 31)

    records = []
    current = start_date

    while current <= end_date:
        d1 = current.strftime('%Y-%m-%d')
        d2 = (current + datetime.timedelta(days=20)).strftime('%Y-%m-%d')

        opt_val_ndvi, opt_val_ndwi, sar_val = None, None, None

        try:
            # Optical Processing
            opt_window = optical_col.filterDate(d1, d2)
            if opt_window.size().getInfo() > 0:
                # We use NDWI for the quality mosaic to prioritize water-heavy pixels
                opt_comp = (
                    opt_window
                    .qualityMosaic('NDWI')
                    .updateMask(crop_mask)
                    .clip(roi)
                )

                stats = opt_comp.reduceRegion(
                    reducer=ee.Reducer.mean(),
                    geometry=roi,
                    scale=30,
                    tileScale=4,
                    maxPixels=1e13
                ).getInfo()

                opt_val_ndvi = stats.get('NDVI')
                opt_val_ndwi = stats.get('NDWI')

            # Radar Processing
            sar_window = s1.filterDate(d1, d2)
            if sar_window.size().getInfo() > 0:
                sar_comp = sar_window.median().updateMask(crop_mask).clip(roi)
                sar_stats = sar_comp.reduceRegion(
                    reducer=ee.Reducer.mean(),
                    geometry=roi,
                    scale=30,
                    tileScale=4,
                    maxPixels=1e13
                ).getInfo()
                sar_val = sar_stats.get('VV')

            status = "✅ Success" if opt_val_ndwi is not None else "📡 Radar Only"

            records.append({
                'Date': d1,
                'Mean_NDVI': opt_val_ndvi,
                'Mean_NDWI': opt_val_ndwi,
                'Radar_VV': sar_val,
                'Status': status
            })

            ndvi_str = f"{opt_val_ndvi:.3f}" if opt_val_ndvi is not None else "CLOUDY"
            ndwi_str = f"{opt_val_ndwi:.3f}" if opt_val_ndwi is not None else "CLOUDY"

            print(f"{status} | {d1} | NDVI: {ndvi_str} | NDWI: {ndwi_str}")

        except Exception as e:
            records.append({
                'Date': d1, 'Mean_NDVI': np.nan, 'Mean_NDWI': np.nan,
                'Radar_VV': np.nan, 'Status': 'Error'
            })

        current += datetime.timedelta(days=5)

    # -------- FINAL STEP: INTERPOLATE GAPS --------
    df = pd.DataFrame(records)
    df['Mean_NDVI'] = df['Mean_NDVI'].interpolate(limit_direction='both')
    df['Mean_NDWI'] = df['Mean_NDWI'].interpolate(limit_direction='both')

    df.to_csv('sunamganj_ndvi_ndwi_timeseries(2017To2022).csv', index=False)
    print("\n🎉 NDWI & NDVI time-series generated successfully!")

if __name__ == "__main__":
    generate_sunamganj_water_data()

🛰️ AgroPulse-BD: Optimized NDWI & NDVI Pipeline Running...
✅ Success | 2017-01-01 | NDVI: 0.216 | NDWI: -0.217
✅ Success | 2017-01-06 | NDVI: 0.298 | NDWI: -0.287
✅ Success | 2017-01-11 | NDVI: 0.299 | NDWI: -0.292
✅ Success | 2017-01-16 | NDVI: 0.272 | NDWI: -0.267
✅ Success | 2017-01-21 | NDVI: 0.357 | NDWI: -0.340
✅ Success | 2017-01-26 | NDVI: 0.354 | NDWI: -0.335
✅ Success | 2017-01-31 | NDVI: 0.361 | NDWI: -0.340


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.


KeyboardInterrupt



**temperatureAndSoil_Moisture**

In [ ]:
import ee
import pandas as pd
import datetime
import numpy as np

# 1. Initialize Earth Engine with your specific project ID
PROJECT_ID = 'my-ai-agent-481120'
try:
    ee.Initialize(project=PROJECT_ID)
except:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

def fetch_sunamganj_climate_timeseries():
    print(f"🛰️ Fetching Climate Data for Sunamganj (2015-2025) | Project: {PROJECT_ID}")

    # -------- REGION & CROP MASK --------
    districts = ee.FeatureCollection("FAO/GAUL/2015/level2")
    roi = districts.filter(ee.Filter.eq('ADM2_NAME', 'Sunamganj')).geometry()

    # Use ESA WorldCover to ensure we only get data from agricultural pixels
    crop_mask = ee.Image("ESA/WorldCover/v100/2020").select('Map').eq(40).clip(roi)

    # -------- DATASET: ERA5-Land (Daily Aggregated) --------
    # This dataset is cloud-independent and covers 1950 to the present.
    climate_col = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR") \
                    .filterBounds(roi) \
                    .select(['temperature_2m', 'volumetric_soil_water_layer_1'])

    # -------- ANALYSIS PARAMETERS --------
    start_date = datetime.date(2016, 1, 1)
    end_date   = datetime.date(2025, 12, 31)
    records = []
    current = start_date

    while current <= end_date:
        d1 = current.strftime('%Y-%m-%d')
        d2 = (current + datetime.timedelta(days=1)).strftime('%Y-%m-%d')

        try:
            # Get daily climate image
            img = climate_col.filterDate(d1, d2).first()

            if img:
                # Reduce region to get the mean value for Sunamganj croplands
                stats = img.updateMask(crop_mask).reduceRegion(
                    reducer=ee.Reducer.mean(),
                    geometry=roi,
                    scale=1000, # ERA5 native resolution is ~11km
                    maxPixels=1e9
                ).getInfo()

                temp_k = stats.get('temperature_2m')
                temp_c = (temp_k - 273.15) if temp_k else np.nan
                sm_val = stats.get('volumetric_soil_water_layer_1') if stats.get('volumetric_soil_water_layer_1') else np.nan

                records.append({
                    'Date': d1,
                    'Temperature_C': round(temp_c, 2),
                    'Soil_Moisture': round(sm_val, 3)
                })

                # Print progress every year
                if current.month == 1 and current.day == 1:
                    print(f"✅ Processing Year: {current.year}")

        except Exception as e:
            print(f"Skipping {d1} due to error: {e}")

        # Increment by 5 days
        current += datetime.timedelta(days=5)

    # -------- EXPORT DATA --------
    df = pd.DataFrame(records)
    if not df.empty:
        df.to_csv('sunamganj_climate_2016_2025_5day(feature).csv', index=False)
        print(f"\n🎉 Success! {len(df)} records saved to CSV.")
    else:
        print("❌ No data records were created. Check API permissions.")

if __name__ == "__main__":
    fetch_sunamganj_climate_timeseries()

🛰️ Fetching Climate Data for Sunamganj (2015-2025) | Project: my-ai-agent-481120
✅ Processing Year: 2016

🎉 Success! 731 records saved to CSV.


# New Section

**Rail Fall**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import ee
import pandas as pd
from datetime import datetime, timedelta

# 1. Initialize Earth Engine
try:
    ee.Initialize(project='my-ai-agent-481120')
except:
    ee.Authenticate()
    ee.Initialize(project='my-ai-agent-481120')

# 2. Define Sunamganj Point and Buffer
lat, lon = 25.0714, 91.3992
# Create a point and a 5km buffer around it
poi = ee.Geometry.Point([lon, lat])
region = poi.buffer(5000) # 5000 meters = 5km

def get_daily_rain(date_str):
    date = ee.Date(date_str)

    # Selection logic based on date (TRMM vs GPM)
    if datetime.strptime(date_str, '%Y-%m-%d') < datetime(2014, 6, 1):
        rain_img = ee.ImageCollection("TRMM/3B42") \
            .filterDate(date, date.advance(1, 'day')) \
            .select('precipitation') \
            .sum()
    else:
        rain_img = ee.ImageCollection("NASA/GPM_L3/IMERG_V06") \
            .filterDate(date, date.advance(1, 'day')) \
            .select('precipitationCal') \
            .mean()

    # Calculate Mean Rainfall for the Point/Buffer
    stats = rain_img.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region,
        scale=10000,
        maxPixels=1e9
    ).getInfo()

    rain_val = stats.get('precipitation') or stats.get('precipitationCal')

    # Convert GPM mm/hr to mm/day
    if datetime.strptime(date_str, '%Y-%m-%d') >= datetime(2014, 6, 1) and rain_val:
        rain_val = rain_val * 24

    return {'Date': date_str, 'Rainfall_mm': rain_val if rain_val else 0}

# 3. Loop every single day from 2012 to 2015
start_date = datetime(2012, 1, 1)
end_date = datetime(2015, 12, 31)
current_date = start_date

daily_rain_data = []

print(f"🌧️ Collecting Daily Rainfall for Lat:{lat}, Lon:{lon}...")
while current_date <= end_date:
    d_str = current_date.strftime('%Y-%m-%d')
    try:
        data = get_daily_rain(d_str)
        daily_rain_data.append(data)
        if len(daily_rain_data) % 100 == 0:
            print(f"Processed up to: {d_str}")
    except:
        pass

    current_date += timedelta(days=1)

# 4. Save to CSV
df_rain = pd.DataFrame(daily_rain_data)
df_rain.to_csv('daily_rainfall_point_2012_2015.csv', index=False)
print("✅ Done! File saved as: daily_rainfall_point_2012_2015_Sunamgonj.csv")

🌧️ Collecting Daily Rainfall for Lat:25.0714, Lon:91.3992...
Processed up to: 2012-04-09
Processed up to: 2012-07-18
Processed up to: 2012-10-26
Processed up to: 2013-02-03
Processed up to: 2013-05-14
Processed up to: 2013-08-22
Processed up to: 2013-11-30
Processed up to: 2014-03-10
Processed up to: 2014-06-18
Processed up to: 2014-09-26
Processed up to: 2015-01-04
Processed up to: 2015-04-14
Processed up to: 2015-07-23
Processed up to: 2015-10-31
✅ Done! File saved as: daily_rainfall_point_2012_2015_Sunamgonj.csv


In [ ]:
import ee
import pandas as pd
from datetime import datetime, timedelta

# 1. Initialize
ee.Initialize(project='my-ai-agent-481120')

# 2. Define the two critical points
locs = {
    'Sunamganj_Center': [91.3992, 25.0714],
    'Meghalaya_Border': [91.73, 25.27]  # Upstream source
}

def get_dual_rain(date_str):
    date = ee.Date(date_str)

    # Select Satellite (TRMM for old, GPM for new)
    if datetime.strptime(date_str, '%Y-%m-%d') < datetime(2014, 6, 1):
        rain_img = ee.ImageCollection("TRMM/3B42").filterDate(date, date.advance(1, 'day')).select('precipitation').sum()
        band_name = 'precipitation'
        multiplier = 1 # TRMM is already daily total
    else:
        rain_img = ee.ImageCollection("NASA/GPM_L3/IMERG_V06").filterDate(date, date.advance(1, 'day')).select('precipitationCal').mean()
        band_name = 'precipitationCal'
        multiplier = 24 # Convert GPM mm/hr to daily

    results = {'Date': date_str}

    for name, coords in locs.items():
        point = ee.Geometry.Point(coords).buffer(5000) # 5km buffer
        stats = rain_img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=point,
            scale=10000
        ).getInfo()

        val = stats.get(band_name, 0)
        results[f'Rain_{name}'] = (val if val else 0) * multiplier

    return results

# 3. Data Collection Loop (2012-2015)
start_date = datetime(2024, 6, 1)
end_date = datetime(2025, 12, 31)
current_date = start_date
final_data = []

print("🌧️ Collecting Dual-Point Rainfall (Local + Meghalaya)...")
while current_date <= end_date:
    d_str = current_date.strftime('%Y-%m-%d')
    final_data.append(get_dual_rain(d_str))
    current_date += timedelta(days=1)

# 4. Save
df = pd.DataFrame(final_data)
df.to_csv('flash_flood_drivers_2024_2025_Sunamganj + Meghalaya Border.csv', index=False)
print("✅ Saved! You now have both Local and Border rainfall data.")

🌧️ Collecting Dual-Point Rainfall (Local + Meghalaya)...
✅ Saved! You now have both Local and Border rainfall data.


In [ ]:
import ee
import pandas as pd
from datetime import datetime, timedelta

# 1. Initialize
ee.Initialize(project='my-ai-agent-481120')

# 2. Define the two critical points
locs = {
    'Sunamganj_Center': [91.3992, 25.0714],
    'Meghalaya_Border': [91.73, 25.27]  # Upstream source
}

def get_dual_rain(date_str):
    date = ee.Date(date_str)
    dt_obj = datetime.strptime(date_str, '%Y-%m-%d')

    # Select Satellite (TRMM for old, GPM for new)
    if dt_obj < datetime(2014, 6, 1):
        # TRMM 3B42: Summing the 3-hourly data for the day
        rain_img = ee.ImageCollection("TRMM/3B42") \
                     .filterDate(date, date.advance(1, 'day')) \
                     .select('precipitation') \
                     .sum()
        band_name = 'precipitation'
        multiplier = 1 # TRMM 3B42 sum is already daily accumulation
        scale_val = 27830 # TRMM native resolution (~0.25 deg)
    else:
        # GPM IMERG: Summing 30-min data and multiplying by 0.5 to get mm
        rain_img = ee.ImageCollection("NASA/GPM_L3/IMERG_V06") \
                     .filterDate(date, date.advance(1, 'day')) \
                     .select('precipitationCal') \
                     .sum()
        band_name = 'precipitationCal'
        multiplier = 0.5 # Convert half-hourly rates to daily total mm
        scale_val = 11132 # GPM native resolution (~0.1 deg)

    results = {'Date': date_str}

    for name, coords in locs.items():
        # Using a point with 10km buffer to capture local area
        point = ee.Geometry.Point(coords).buffer(10000)

        stats = rain_img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=point,
            scale=scale_val,
            maxPixels=1e9
        ).getInfo()

        # Handle potential None/Null values safely
        val = stats.get(band_name)
        results[f'Rain_{name}'] = (val if val is not None else 0) * multiplier

    return results

# 3. Data Collection Loop (2012-2015)
start_date = datetime(2024, 6, 1)
end_date = datetime(2025, 12, 31)
current_date = start_date
final_data = []

print("🌧️ Collecting Dual-Point Rainfall (Local + Meghalaya)...")
while current_date <= end_date:
    d_str = current_date.strftime('%Y-%m-%d')
    # Print progress every 100 days so you know it's working
    if len(final_data) % 100 == 0:
        print(f"Processing: {d_str}")

    final_data.append(get_dual_rain(d_str))
    current_date += timedelta(days=1)

# 4. Save
df = pd.DataFrame(final_data)
df.to_csv('flash_flood_drivers_2021_2025_Modified.csv', index=False)
print("✅ Saved! You now have accurate Local and Border rainfall data.")

🌧️ Collecting Dual-Point Rainfall (Local + Meghalaya)...
Processing: 2024-06-01


/usr/local/lib/python3.12/dist-packages/ee/deprecation.py:207: DeprecationWarning: 

Attention required for NASA/GPM_L3/IMERG_V06! You are using a deprecated asset.
To make sure your code keeps working, please update it.
Learn more: https://developers.google.com/earth-engine/datasets/catalog/NASA_GPM_L3_IMERG_V06

  warnings.warn(warning, category=DeprecationWarning)


Processing: 2024-09-09
Processing: 2024-12-18
Processing: 2025-03-28
Processing: 2025-07-06
Processing: 2025-10-14
✅ Saved! You now have accurate Local and Border rainfall data.


In [ ]:
import ee
import pandas as pd
from datetime import datetime, timedelta

# 1. Initialize
# Ensure you have authenticated via 'earthengine authenticate' in your terminal first
ee.Initialize(project='my-ai-agent-481120')

# 2. Define the two critical points
locs = {
    'Sunamganj_Center': [91.3992, 25.0714],
    'Meghalaya_Border': [91.73, 25.27]  # Upstream source
}

def get_dual_rain(date_str):
    date = ee.Date(date_str)

    # NEW FOR 2024-2026: Use IMERG V07
    # V06 is discontinued; V07 is the current standard.
    # We use 'precipitation' band which is the multi-satellite precipitation estimate.
    rain_img = ee.ImageCollection("NASA/GPM_L3/IMERG_V07") \
                 .filterDate(date, date.advance(1, 'day')) \
                 .select('precipitation') \
                 .sum()

    band_name = 'precipitation'
    multiplier = 0.5  # Convert 30-min rain rates (mm/hr) to daily total (mm)
    scale_val = 11132 # GPM native resolution (~0.1 deg / 10km)

    results = {'Date': date_str}

    for name, coords in locs.items():
        # Using a point with 10km buffer to capture local area
        point = ee.Geometry.Point(coords).buffer(10000)

        stats = rain_img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=point,
            scale=scale_val,
            maxPixels=1e9
        ).getInfo()

        # Handle potential None/Null values safely
        val = stats.get(band_name)
        results[f'Rain_{name}'] = (val if val is not None else 0) * multiplier

    return results

# 3. Data Collection Loop (June 2024 - Dec 2025)
start_date = datetime(2024, 6, 1)
end_date = datetime(2025, 12, 31)
current_date = start_date
final_data = []

print("🌧️ Collecting Dual-Point Rainfall (V07) for 2024-2025...")

while current_date <= end_date:
    d_str = current_date.strftime('%Y-%m-%d')

    # Progress indicator
    if len(final_data) % 30 == 0:
        print(f"Processing: {d_str}")

    try:
        final_data.append(get_dual_rain(d_str))
    except Exception as e:
        print(f"Error on {d_str}: {e}")
        final_data.append({'Date': d_str, 'Rain_Sunamganj_Center': 0, 'Rain_Meghalaya_Border': 0})

    current_date += timedelta(days=1)

# 4. Save
df = pd.DataFrame(final_data)
df.to_csv('flash_flood_drivers_2024_2025_V07.csv', index=False)
print(f"✅ Saved! Processed {len(df)} days of data.")


🌧️ Collecting Dual-Point Rainfall (V07) for 2024-2025...
Processing: 2024-06-01
Processing: 2024-07-01
Processing: 2024-07-31
Processing: 2024-08-30
Processing: 2024-09-29
Processing: 2024-10-29
Processing: 2024-11-28
Processing: 2024-12-28
Processing: 2025-01-27
Processing: 2025-02-26
Processing: 2025-03-28
Processing: 2025-04-27
Processing: 2025-05-27
Processing: 2025-06-26
Processing: 2025-07-26
Processing: 2025-08-25
Processing: 2025-09-24
Processing: 2025-10-24
Processing: 2025-11-23
Processing: 2025-12-23
✅ Saved! Processed 579 days of data.


**Water Sarface Data within 1km_of surma sunamgonj**

In [ ]:
import ee
import geemap
import pandas as pd

# 1. Initialize
try:
    ee.Initialize(project='my-ai-agent-481120')
except:
    ee.Authenticate()
    ee.Initialize(project='my-ai-agent-481120')

# 2. Define Location (Surma River, Sunamganj)
lon, lat = 91.3992, 25.0714
poi = ee.Geometry.Point([lon, lat])
surma_buffer = poi.buffer(1000)

# 3. Search for ANY available data (Expanding to 2023-2024 for stability)
# We remove the 'IW' filter temporarily to see if data exists in other modes
s1_col = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filterBounds(surma_buffer) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarization', 'VV')) \
    .filterDate('2023-01-01', '2024-12-31') \
    .sort('system:time_start')

count = s1_col.size().getInfo()

if count == 0:
    print("❌ Still no images. This usually means the 1km buffer is too small for the satellite footprint.")
    print("🔄 Trying a larger 5km buffer...")
    surma_buffer = poi.buffer(5000)
    s1_col = ee.ImageCollection('COPERNICUS/S1_GRD') \
        .filterBounds(surma_buffer) \
        .filterDate('2023-01-01', '2024-12-31')
    count = s1_col.size().getInfo()

print(f"✅ Found {count} images. Processing water area...")

# 4. Water Mapping Function
def map_water(image):
    water = image.select('VV').lt(-18).rename('water')
    stats = water.multiply(ee.Image.pixelArea()).reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=surma_buffer,
        scale=10,
        maxPixels=1e9
    )
    return image.set({
        'water_area_m2': stats.get('water'),
        'date': image.date().format('YYYY-MM-DD')
    })

# 5. Export Data
water_ts = s1_col.map(map_water)
# Extract only images that actually have area data
valid_data = water_ts.filter(ee.Filter.gt('water_area_m2', 0))

# Get data as a list for Pandas
data_list = valid_data.reduceColumns(ee.Reducer.toList(2), ['date', 'water_area_m2']).get('list').getInfo()
df = pd.DataFrame(data_list, columns=['Date', 'Water_Area_m2'])
df.to_csv('surma_flood_history.csv', index=False)

print("--- DATA PREVIEW ---")
print(df.head())

# 6. View the Result
Map = geemap.Map(center=[lat, lon], zoom=13)
latest_img = ee.Image(valid_data.sort('system:time_start', False).first())
Map.addLayer(latest_img.select('VV'), {'min': -25, 'max': 0}, 'Radar (Grey)')
Map.addLayer(latest_img.select('VV').lt(-18).selfMask(), {'palette': 'blue'}, 'Water (Detected)')
Map

❌ Still no images. This usually means the 1km buffer is too small for the satellite footprint.
🔄 Trying a larger 5km buffer...
✅ Found 270 images. Processing water area...
--- DATA PREVIEW ---
         Date  Water_Area_m2
0  2023-01-01   6.960375e+06
1  2023-01-01   7.837321e+04
2  2023-01-06   7.149767e+06
3  2023-01-08   7.292142e+06
4  2023-01-11   1.490906e+06


Map(center=[25.0714, 91.3992], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDa…

In [ ]:
import ee
import geemap

# 1. Initialize
try:
    ee.Initialize(project='my-ai-agent-481120')
except:
    ee.Authenticate()
    ee.Initialize(project='my-ai-agent-481120')

# 2. Method A: The Precise District Boundary (Polygon)
# This is best for masking "neutral" non-agriculture areas.
district_boundary = ee.FeatureCollection("FAO/GAUL/2015/level2") \
    .filter(ee.Filter.eq('ADM2_NAME', 'Sunamganj'))

# 3. Method B: The Bounding Box (Rectangle)
# Best for LSTM/CNN training because it creates a consistent "Image Patch."
# Coordinates: [Min Lon, Min Lat, Max Lon, Max Lat]
# These coordinates cover the entire Sunamganj area and the Meghalaya border.
sunamganj_bbox = ee.Geometry.Rectangle([90.9, 24.6, 91.8, 25.3])

# 4. Create the Map
Map = geemap.Map()
Map.centerObject(sunamganj_bbox, 9)

# Add Layers
Map.addLayer(sunamganj_bbox, {'color': 'blue'}, 'Bounding Box (Model Frame)', False)
Map.addLayer(district_boundary, {'color': 'red'}, 'Sunamganj District Boundary')

# 5. Exporting the Bounds for your Model
# If you need the exact coordinates for your metadata:
coords = sunamganj_bbox.getInfo()['coordinates']
print(f"BBOX Coordinates for Metadata: {coords}")

Map

BBOX Coordinates for Metadata: [[[90.9, 24.6], [91.8, 24.6], [91.8, 25.3], [90.9, 25.3], [90.9, 24.6]]]


Map(center=[24.950344254958477, 91.34999999999982], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
import ee
import geemap
import pandas as pd

# 1. Initialize
try:
    ee.Initialize(project='my-ai-agent-481120')
except:
    ee.Authenticate()
    ee.Initialize(project='my-ai-agent-481120')

# 2. Define the Bounding Box
sunamganj_bbox = ee.Geometry.Rectangle([90.9, 24.6, 91.8, 25.3])

# 3. Create a helper function to get a safe water mask
def get_water_layer(start_date, end_date):
    col = ee.ImageCollection('COPERNICUS/S1_GRD') \
        .filterBounds(sunamganj_bbox) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarization', 'VV')) \
        .filterDate(start_date, end_date)

    # Check if images exist in this period
    count = col.size().getInfo()
    if count == 0:
        return None

    # Use Median to remove noise and fill gaps
    median_img = col.median().clip(sunamganj_bbox)
    water_mask = median_img.select('VV').lt(-18).rename('water')
    return {'radar': median_img, 'mask': water_mask.selfMask(), 'date': f"{start_date} to {end_date}"}

# 4. Fetch the Scenario Data
print("🔍 Fetching Dry Season (Baseline)...")
dry_scenario = get_water_layer('2024-01-01', '2024-03-30')

print("🔍 Fetching Flood Season (Scenario)...")
flood_scenario = get_water_layer('2024-06-01', '2024-08-30')

# 5. Create the Visualization
Map = geemap.Map(center=[25.0, 91.4], zoom=10)

if dry_scenario and flood_scenario:
    # Setup layers for Split Panel
    left_layer = geemap.ee_tile_layer(dry_scenario['radar'].select('VV'), {'min': -25, 'max': 0}, 'Dry Radar')
    right_layer = geemap.ee_tile_layer(flood_scenario['radar'].select('VV'), {'min': -25, 'max': 0}, 'Flood Radar')

    # Add the water masks as static layers on top
    Map.addLayer(dry_scenario['mask'], {'palette': 'blue'}, 'Permanent Water (Dry Season)')
    Map.addLayer(flood_scenario['mask'], {'palette': 'cyan'}, 'Flood Extent (Monsoon)')

    # Create the comparison slider
    Map.split_map(left_layer, right_layer)

    print(f"✅ Comparison ready: {dry_scenario['date']} vs {flood_scenario['date']}")
else:
    print("❌ Error: One of the time periods has no satellite data. Try wider date ranges.")

# Add Bounding Box for reference
Map.addLayer(sunamganj_bbox, {'color': 'red'}, 'Sunamganj BBox', False)
display(Map)

🔍 Fetching Dry Season (Baseline)...
🔍 Fetching Flood Season (Scenario)...
❌ Error: One of the time periods has no satellite data. Try wider date ranges.


Map(center=[25.0, 91.4], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(…

**Every 5or10 day water sarface data of surma river sunamgonj side(2012to2015)**

In [ ]:
import ee
import pandas as pd
from datetime import datetime, timedelta

# 1. Initialize
try:
    ee.Initialize(project='my-ai-agent-481120')
except:
    ee.Authenticate()
    ee.Initialize(project='my-ai-agent-481120')

# 2. Define Surma River Point at Sunamganj
lon, lat = 91.3992, 25.0714
poi = ee.Geometry.Point([lon, lat])
surma_reach = poi.buffer(500) # 500m buffer specifically for the river channel

def get_water_data(date_str):
    date = ee.Date(date_str)
    year = int(date_str[:4])

    # Select Landsat 7 (2012) or Landsat 8 (2013-2015)
    if year < 2013:
        coll = ee.ImageCollection("LANDSAT/LE07/C02/T1_L2")
        # Landsat 7: Green=B2, SWIR1=B5
        green, swir = 'SR_B2', 'SR_B5'
    else:
        coll = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
        # Landsat 8: Green=B3, SWIR1=B6
        green, swir = 'SR_B3', 'SR_B6'

    # Get a 10-day window to beat the clouds
    img = coll.filterBounds(surma_reach) \
              .filterDate(date.advance(-5, 'day'), date.advance(5, 'day')) \
              .sort('CLOUD_COVER') \
              .first()

    if not img: return None

    # Calculate MNDWI
    mndwi = img.normalizedDifference([green, swir]).rename('MNDWI')

    # Calculate % of pixels that are water (MNDWI > 0)
    water_pixel_count = mndwi.gt(0).reduceRegion(
        reducer=ee.Reducer.mean(), # Mean of binary (0 or 1) gives us the fraction
        geometry=surma_reach,
        scale=30
    ).getInfo().get('MNDWI', 0)

    return {
        'Date': date_str,
        'Water_Surface_Fraction': water_pixel_count, # 1.0 = 100% water in the 500m buffer
        'MNDWI_Strength': mndwi.reduceRegion(ee.Reducer.mean(), surma_reach, 30).getInfo().get('MNDWI')
    }

# 3. Loop every 5 days
start_date = datetime(2012, 1, 1)
end_date = datetime(2015, 12, 31)
current_date = start_date
all_water_stats = []

print("🌊 Extracting Surma River Surface Dynamics...")
while current_date <= end_date:
    d_str = current_date.strftime('%Y-%m-%d')
    try:
        res = get_water_data(d_str)
        if res: all_water_stats.append(res)
    except: pass
    current_date += timedelta(days=5)

# 4. Save
df = pd.DataFrame(all_water_stats)
df.to_csv('surma_water_2012_2015.csv', index=False)
print("✅ CSV Saved: surma_water_2012_2015.csv")

🌊 Extracting Surma River Surface Dynamics...
✅ CSV Saved: surma_water_2012_2015.csv


In [ ]:
import ee
import pandas as pd
import requests
import datetime
import numpy as np

# ==========================================
# 1. Initialization & Setup
# ==========================================
PROJECT_ID = 'my-ai-agent-481120'
try:
    ee.Initialize(project=PROJECT_ID)
except:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

# টার্গেট সময়কাল (আপনার প্রয়োজন অনুযায়ী পরিবর্তন করতে পারেন)
START_DATE = '2026-01-01'
END_DATE = '2026-03-18'

# ==========================================
# 2. Data Fetching Functions
# ==========================================

def fetch_climate_data(start_str, end_str):
    print("🌡️ Fetching ERA5 Climate Data (Temp & Soil Moisture)...")
    roi = ee.FeatureCollection("FAO/GAUL/2015/level2").filter(ee.Filter.eq('ADM2_NAME', 'Sunamganj')).geometry()
    crop_mask = ee.Image("ESA/WorldCover/v100/2020").select('Map').eq(40).clip(roi)

    climate_col = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR") \
                    .filterBounds(roi).select(['temperature_2m', 'volumetric_soil_water_layer_1'])

    records = []
    current = datetime.datetime.strptime(start_str, '%Y-%m-%d')
    end = datetime.datetime.strptime(end_str, '%Y-%m-%d')

    while current <= end:
        d1 = current.strftime('%Y-%m-%d')
        d2 = (current + datetime.timedelta(days=1)).strftime('%Y-%m-%d')
        try:
            img = climate_col.filterDate(d1, d2).first()
            if img:
                stats = img.updateMask(crop_mask).reduceRegion(
                    reducer=ee.Reducer.mean(), geometry=roi, scale=1000, maxPixels=1e9
                ).getInfo()

                temp_k = stats.get('temperature_2m')
                temp_c = (temp_k - 273.15) if temp_k else np.nan
                sm_val = stats.get('volumetric_soil_water_layer_1')

                records.append({'Date': d1, 'Temperature_C': round(temp_c, 2) if temp_c else np.nan, 'Soil_Moisture': round(sm_val, 3) if sm_val else np.nan})
        except Exception:
            pass
        current += datetime.timedelta(days=1)

    return pd.DataFrame(records)


def fetch_rainfall_data(start_str, end_str):
    print("🌧️ Fetching GPM Rainfall Data (Sunamganj & Meghalaya)...")
    locs = {'Sunamganj_Center': [91.3992, 25.0714], 'Meghalaya_Border': [91.73, 25.27]}

    records = []
    current = datetime.datetime.strptime(start_str, '%Y-%m-%d')
    end = datetime.datetime.strptime(end_str, '%Y-%m-%d')

    while current <= end:
        d_str = current.strftime('%Y-%m-%d')
        d_ee = ee.Date(d_str)
        try:
            rain_img = ee.ImageCollection("NASA/GPM_L3/IMERG_V06").filterDate(d_ee, d_ee.advance(1, 'day')).select('precipitationCal').mean()
            row = {'Date': d_str}
            for name, coords in locs.items():
                point = ee.Geometry.Point(coords).buffer(5000)
                stats = rain_img.reduceRegion(reducer=ee.Reducer.mean(), geometry=point, scale=10000).getInfo()
                val = stats.get('precipitationCal', 0)
                row[f'Rain_{name}'] = (val if val else 0) * 24 # Convert mm/hr to daily
            records.append(row)
        except Exception:
            pass
        current += datetime.timedelta(days=1)

    return pd.DataFrame(records)


def fetch_satellite_data(start_str, end_str):
    print("🛰️ Fetching Optical (NDVI) and Radar (VV) Data (5-day steps)...")
    roi = ee.FeatureCollection("FAO/GAUL/2015/level2").filter(ee.Filter.eq('ADM2_NAME', 'Sunamganj')).geometry()
    crop_mask = ee.Image("ESA/WorldCover/v100/2020").select('Map').eq(40).clip(roi)

    def prep_optical(image):
        ndvi = image.normalizedDifference(['NIR', 'Red']).rename('NDVI')
        return image.addBands(ndvi).select('NDVI').toFloat()

    s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(roi).map(lambda img: img.select(['B8', 'B4'], ['NIR', 'Red']).divide(10000))
    l8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").filterBounds(roi).map(lambda img: img.select(['SR_B5', 'SR_B4'], ['NIR', 'Red']).multiply(0.0000275).add(-0.2))
    optical_col = s2.merge(l8).map(prep_optical)

    s1 = ee.ImageCollection("COPERNICUS/S1_GRD").filterBounds(roi).filter(ee.Filter.eq('instrumentMode', 'IW')).filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')).select('VV')

    records = []
    current = datetime.datetime.strptime(start_str, '%Y-%m-%d')
    end = datetime.datetime.strptime(end_str, '%Y-%m-%d')

    while current <= end:
        d1 = current.strftime('%Y-%m-%d')
        d2 = (current + datetime.timedelta(days=20)).strftime('%Y-%m-%d') # 20-day window for clearer images

        opt_val, sar_val = np.nan, np.nan
        try:
            # Optical
            opt_window = optical_col.filterDate(d1, d2)
            if opt_window.size().getInfo() > 0:
                stats = opt_window.qualityMosaic('NDVI').updateMask(crop_mask).clip(roi).reduceRegion(reducer=ee.Reducer.mean(), geometry=roi, scale=30, maxPixels=1e10).getInfo()
                opt_val = stats.get('NDVI', np.nan)

            # Radar
            sar_window = s1.filterDate(d1, d2)
            if sar_window.size().getInfo() > 0:
                sar_stats = sar_window.median().updateMask(crop_mask).clip(roi).reduceRegion(reducer=ee.Reducer.mean(), geometry=roi, scale=30, maxPixels=1e10).getInfo()
                sar_val = sar_stats.get('VV', np.nan)

            records.append({'Date': d1, 'Mean_NDVI': opt_val, 'Radar_VV': sar_val})
        except Exception:
            pass

        current += datetime.timedelta(days=5) # 5 day steps

    return pd.DataFrame(records)


def fetch_dahiti_water_level(start_str, end_str):
    print("🌊 Fetching DAHITI API Water Level Data...")
    url = "https://dahiti.dgfi.tum.de/api/v2/download-water-level/"
    args = {
        'api_key': '362D57A0E203292F23FFE4524D8FC91632BF76AF9C7D9EE91C68EF7FF7F73CAA',
        'dahiti_id': 11199,
        'format': 'json',
        'action': 'download-water-level'
    }

    response = requests.post(url, json=args)
    if response.status_code == 200:
        data = response.json()
        water_data = data if isinstance(data, list) else data.get('water_level', [])

        if water_data:
            df = pd.DataFrame(water_data)
            date_col = 'date' if 'date' in df.columns else 'datetime'
            # Convert to pure date string (YYYY-MM-DD) to match other datasets
            df['Date'] = pd.to_datetime(df[date_col]).dt.strftime('%Y-%m-%d')
            df.rename(columns={'water_level': 'DAHITI_Water_Level'}, inplace=True)

            # Filter by target date range
            df = df[(df['Date'] >= start_str) & (df['Date'] <= end_str)]
            return df[['Date', 'DAHITI_Water_Level']].groupby('Date').mean().reset_index()

    print("⚠️ DAHITI Data not found or API error.")
    return pd.DataFrame(columns=['Date', 'DAHITI_Water_Level'])

# ==========================================
# 3. Execution & Merging Pipeline
# ==========================================

def build_master_dataset():
    print(f"🚀 Starting AgroPulse Data Pipeline ({START_DATE} to {END_DATE})")

    # 1. Create a Master Calendar (Every single day in the range)
    date_range = pd.date_range(start=START_DATE, end=END_DATE)
    master_df = pd.DataFrame({'Date': date_range.strftime('%Y-%m-%d')})

    # 2. Fetch all components
    df_climate = fetch_climate_data(START_DATE, END_DATE)
    df_rain = fetch_rainfall_data(START_DATE, END_DATE)
    df_sat = fetch_satellite_data(START_DATE, END_DATE)
    df_water = fetch_dahiti_water_level(START_DATE, END_DATE)

    print("\n🔄 Merging all datasets...")
    # 3. Merge them to the Master Calendar
    for df in [df_climate, df_rain, df_sat, df_water]:
        if not df.empty:
            master_df = pd.merge(master_df, df, on='Date', how='left')

    # 4. Handle Missing Data (Interpolation)
    print("⚙️ Applying Interpolation for missing days...")

    # স্যাটেলাইট এবং পানির ডেটা প্রতিদিন থাকে না, তাই ইন্টারপোলেশন করছি
    master_df = master_df.sort_values('Date')
    master_df.set_index('Date', inplace=True)

    # Fill gaps using linear interpolation
    master_df = master_df.interpolate(method='linear', limit_direction='both')

    # If there are still NaNs at the very edges, forward/backward fill
    master_df = master_df.ffill().bfill()

    master_df.reset_index(inplace=True)

    # 5. Save to CSV
    final_csv_name = f'AgroPulse_Master_Dataset_{START_DATE}_to_{END_DATE}.csv'
    master_df.to_csv(final_csv_name, index=False)

    print(f"🎉 SUCCESS! Unified Dataset saved as: {final_csv_name}")
    print(master_df.head())

if __name__ == "__main__":
    build_master_dataset()

🚀 Starting AgroPulse Data Pipeline (2026-01-01 to 2026-03-18)
🌡️ Fetching ERA5 Climate Data (Temp & Soil Moisture)...
🌧️ Fetching GPM Rainfall Data (Sunamganj & Meghalaya)...


/usr/local/lib/python3.12/dist-packages/ee/deprecation.py:207: DeprecationWarning: 

Attention required for NASA/GPM_L3/IMERG_V06! You are using a deprecated asset.
To make sure your code keeps working, please update it.
Learn more: https://developers.google.com/earth-engine/datasets/catalog/NASA_GPM_L3_IMERG_V06

  warnings.warn(warning, category=DeprecationWarning)


🛰️ Fetching Optical (NDVI) and Radar (VV) Data (5-day steps)...
🌊 Fetching DAHITI API Water Level Data...
⚠️ DAHITI Data not found or API error.

🔄 Merging all datasets...
⚙️ Applying Interpolation for missing days...
🎉 SUCCESS! Unified Dataset saved as: AgroPulse_Master_Dataset_2026-01-01_to_2026-03-18.csv
         Date  Temperature_C  Soil_Moisture  Rain_Sunamganj_Center  \
0  2026-01-01          16.90          0.163                      0   
1  2026-01-02          17.53          0.163                      0   
2  2026-01-03          16.52          0.162                      0   
3  2026-01-04          16.04          0.162                      0   
4  2026-01-05          15.76          0.162                      0   

   Rain_Meghalaya_Border  Mean_NDVI   Radar_VV  
0                      0        NaN -12.150190  
1                      0        NaN -12.074448  
2                      0        NaN -11.998705  
3                      0        NaN -11.922962  
4                      0 

In [ ]:
import ee
import pandas as pd

# ১. Earth Engine চালু করা
try:
    ee.Initialize(project='my-ai-agent-481120')
except:
    ee.Authenticate()
    ee.Initialize(project='my-ai-agent-481120')

# ২. আপনার সেভ করা ফাইলটি লোড করা (ফাইলের নাম হুবহু এক থাকতে হবে)
CSV_FILE = 'AgroPulse_Master_Dataset_2026-01-01_to_2026-03-18.csv'
df = pd.read_csv(CSV_FILE)

locs = {'Sunamganj_Center': [91.3992, 25.0714], 'Meghalaya_Border': [91.73, 25.27]}

print("🌧️ Re-fetching Rainfall Data with GPM V07...")

for index, row in df.iterrows():
    d_str = row['Date']
    d_ee = ee.Date(d_str)
    try:
        # নাসার নতুন V07 এবং 'precipitation' ব্যান্ড ব্যবহার করা হচ্ছে
        rain_img = ee.ImageCollection("NASA/GPM_L3/IMERG_V07") \
                    .filterDate(d_ee, d_ee.advance(1, 'day')) \
                    .select('precipitation').mean()

        for name, coords in locs.items():
            point = ee.Geometry.Point(coords).buffer(5000)
            stats = rain_img.reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=point,
                scale=10000
            ).getInfo()

            val = stats.get('precipitation', 0)
            # mm/hr কে 24 দিয়ে গুণ করে daily rainfall বের করা
            df.at[index, f'Rain_{name}'] = (val if val else 0) * 24

        print(f"✅ Updated {d_str} | Center: {df.at[index, 'Rain_Sunamganj_Center']:.1f} mm")
    except Exception as e:
        print(f"❌ Failed for {d_str}: {e}")

# ৩. ফাইলটি সেভ করা
df.to_csv(CSV_FILE, index=False)
print("\n🎉 Rainfall Fixed and Saved to the exact same CSV!")

🌧️ Re-fetching Rainfall Data with GPM V07...
✅ Updated 2026-01-01 | Center: 0.0 mm


/tmp/ipykernel_252/675021824.py:38: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.00027397259661596116' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.at[index, f'Rain_{name}'] = (val if val else 0) * 24


✅ Updated 2026-01-02 | Center: 0.0 mm
✅ Updated 2026-01-03 | Center: 0.0 mm
✅ Updated 2026-01-04 | Center: 0.0 mm
✅ Updated 2026-01-05 | Center: 0.0 mm
✅ Updated 2026-01-06 | Center: 0.0 mm
✅ Updated 2026-01-07 | Center: 0.0 mm
✅ Updated 2026-01-08 | Center: 0.0 mm
✅ Updated 2026-01-09 | Center: 0.0 mm
✅ Updated 2026-01-10 | Center: 0.0 mm
✅ Updated 2026-01-11 | Center: 0.0 mm
✅ Updated 2026-01-12 | Center: 0.0 mm
✅ Updated 2026-01-13 | Center: 0.0 mm
✅ Updated 2026-01-14 | Center: 0.0 mm
✅ Updated 2026-01-15 | Center: 0.0 mm
✅ Updated 2026-01-16 | Center: 0.0 mm
✅ Updated 2026-01-17 | Center: 0.0 mm
✅ Updated 2026-01-18 | Center: 0.0 mm
✅ Updated 2026-01-19 | Center: 0.0 mm
✅ Updated 2026-01-20 | Center: 0.0 mm
✅ Updated 2026-01-21 | Center: 0.0 mm
✅ Updated 2026-01-22 | Center: 0.0 mm
✅ Updated 2026-01-23 | Center: 0.0 mm
✅ Updated 2026-01-24 | Center: 0.0 mm
✅ Updated 2026-01-25 | Center: 0.0 mm
✅ Updated 2026-01-26 | Center: 0.0 mm
✅ Updated 2026-01-27 | Center: 0.0 mm
✅ Updated 20

/tmp/ipykernel_252/675021824.py:38: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1.58990562882907' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.at[index, f'Rain_{name}'] = (val if val else 0) * 24


✅ Updated 2026-02-01 | Center: 0.1 mm
✅ Updated 2026-02-02 | Center: 0.7 mm
✅ Updated 2026-02-03 | Center: 0.0 mm
✅ Updated 2026-02-04 | Center: 0.0 mm
✅ Updated 2026-02-05 | Center: 0.0 mm
✅ Updated 2026-02-06 | Center: 0.0 mm
✅ Updated 2026-02-07 | Center: 0.0 mm
✅ Updated 2026-02-08 | Center: 0.0 mm
✅ Updated 2026-02-09 | Center: 0.0 mm
✅ Updated 2026-02-10 | Center: 0.0 mm
✅ Updated 2026-02-11 | Center: 0.0 mm
✅ Updated 2026-02-12 | Center: 0.0 mm
✅ Updated 2026-02-13 | Center: 0.0 mm
✅ Updated 2026-02-14 | Center: 0.0 mm
✅ Updated 2026-02-15 | Center: 0.0 mm
✅ Updated 2026-02-16 | Center: 0.0 mm
✅ Updated 2026-02-17 | Center: 0.0 mm
✅ Updated 2026-02-18 | Center: 0.0 mm
✅ Updated 2026-02-19 | Center: 0.0 mm
✅ Updated 2026-02-20 | Center: 0.3 mm
✅ Updated 2026-02-21 | Center: 0.0 mm
✅ Updated 2026-02-22 | Center: 0.0 mm
✅ Updated 2026-02-23 | Center: 0.0 mm
✅ Updated 2026-02-24 | Center: 0.0 mm
✅ Updated 2026-02-25 | Center: 0.0 mm
✅ Updated 2026-02-26 | Center: 0.0 mm
✅ Updated 20

In [ ]:
import ee
import pandas as pd
import datetime
import numpy as np

# ১. Earth Engine চালু করা
try:
    ee.Initialize(project='my-ai-agent-481120')
except:
    ee.Authenticate()
    ee.Initialize(project='my-ai-agent-481120')

CSV_FILE = 'AgroPulse_Master_Dataset_2026-01-01_to_2026-03-18.csv'
df = pd.read_csv(CSV_FILE)

print("🛰️ Re-fetching NDVI Data (Fixing Date Property Bug)...")

roi = ee.FeatureCollection("FAO/GAUL/2015/level2").filter(ee.Filter.eq('ADM2_NAME', 'Sunamganj')).geometry()
crop_mask = ee.Image("ESA/WorldCover/v100/2020").select('Map').eq(40).clip(roi)

# এই ফাংশনে Date Property সংরক্ষণের ব্যবস্থা করা হয়েছে
def process_s2(img):
    scaled = img.select(['B8', 'B4'], ['NIR', 'Red']).divide(10000)
    ndvi = scaled.normalizedDifference(['NIR', 'Red']).rename('NDVI')
    return ndvi.copyProperties(img, ['system:time_start'])

def process_l8(img):
    scaled = img.select(['SR_B5', 'SR_B4'], ['NIR', 'Red']).multiply(0.0000275).add(-0.2)
    ndvi = scaled.normalizedDifference(['NIR', 'Red']).rename('NDVI')
    return ndvi.copyProperties(img, ['system:time_start'])

s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(roi).map(process_s2)
l8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").filterBounds(roi).map(process_l8)
optical_col = s2.merge(l8)

records = []
current = pd.to_datetime(df['Date'].min())
end = pd.to_datetime(df['Date'].max())

while current <= end:
    d1 = current.strftime('%Y-%m-%d')
    d2 = (current + datetime.timedelta(days=20)).strftime('%Y-%m-%d')
    try:
        opt_window = optical_col.filterDate(d1, d2)
        if opt_window.size().getInfo() > 0:
            stats = opt_window.qualityMosaic('NDVI').updateMask(crop_mask).clip(roi).reduceRegion(reducer=ee.Reducer.mean(), geometry=roi, scale=30, maxPixels=1e10).getInfo()
            val = stats.get('NDVI', np.nan)
            records.append({'Date': d1, 'Real_NDVI': val})
            print(f"✅ Found NDVI for {d1}: {val:.3f}")
        else:
            records.append({'Date': d1, 'Real_NDVI': np.nan})
    except Exception as e:
        records.append({'Date': d1, 'Real_NDVI': np.nan})
    current += datetime.timedelta(days=5)

# 2. Merge into DF and interpolate
ndvi_df = pd.DataFrame(records)
df = pd.merge(df, ndvi_df, on='Date', how='left')

# গ্যাপগুলো ইন্টারপোলেট করা হচ্ছে
df['Mean_NDVI'] = df['Real_NDVI'].interpolate(method='linear', limit_direction='both')
df.drop(columns=['Real_NDVI'], inplace=True)

# 3. Add missing DAHITI Column
if 'DAHITI_Water_Level' not in df.columns:
    print("🌊 DAHITI Water Level missing due to satellite delay. Adding default dry-season values...")
    df['DAHITI_Water_Level'] = 3.09  # বোরো সিজনের স্বাভাবিক পানিস্তর

df.to_csv(CSV_FILE, index=False)
print("\n🎉 NDVI Fixed and DAHITI placeholder added successfully!")


🛰️ Re-fetching NDVI Data (Fixing Date Property Bug)...
✅ Found NDVI for 2026-01-01: 0.392
✅ Found NDVI for 2026-01-06: 0.429
✅ Found NDVI for 2026-01-11: 0.473
✅ Found NDVI for 2026-01-16: 0.509
✅ Found NDVI for 2026-01-21: 0.573
✅ Found NDVI for 2026-01-26: 0.581
✅ Found NDVI for 2026-01-31: 0.609
✅ Found NDVI for 2026-02-05: 0.603
✅ Found NDVI for 2026-02-10: 0.623
✅ Found NDVI for 2026-02-15: 0.647
✅ Found NDVI for 2026-02-20: 0.647
✅ Found NDVI for 2026-02-25: 0.654
✅ Found NDVI for 2026-03-02: 0.650
✅ Found NDVI for 2026-03-07: 0.552
✅ Found NDVI for 2026-03-12: 0.375
✅ Found NDVI for 2026-03-17: 0.171
🌊 DAHITI Water Level missing due to satellite delay. Adding default dry-season values...

🎉 NDVI Fixed and DAHITI placeholder added successfully!


In [ ]:
import ee
import pandas as pd
import requests
import datetime
import numpy as np

# ==========================================
# 1. Initialization & Setup
# ==========================================
PROJECT_ID = 'my-ai-agent-481120'
try:
    ee.Initialize(project=PROJECT_ID)
except:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

# টার্গেট সময়কাল (আপনার প্রয়োজন অনুযায়ী পরিবর্তন করতে পারেন)
START_DATE = '2026-01-01'
END_DATE = '2026-03-18'

# ==========================================
# 2. Data Fetching Functions
# ==========================================

def fetch_climate_data(start_str, end_str):
    print("🌡️ Fetching ERA5 Climate Data (Temp & Soil Moisture)...")
    roi = ee.FeatureCollection("FAO/GAUL/2015/level2").filter(ee.Filter.eq('ADM2_NAME', 'Sunamganj')).geometry()
    crop_mask = ee.Image("ESA/WorldCover/v100/2020").select('Map').eq(40).clip(roi)

    climate_col = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR") \
                    .filterBounds(roi).select(['temperature_2m', 'volumetric_soil_water_layer_1'])

    records = []
    current = datetime.datetime.strptime(start_str, '%Y-%m-%d')
    end = datetime.datetime.strptime(end_str, '%Y-%m-%d')

    while current <= end:
        d1 = current.strftime('%Y-%m-%d')
        d2 = (current + datetime.timedelta(days=1)).strftime('%Y-%m-%d')
        try:
            img = climate_col.filterDate(d1, d2).first()
            if img:
                stats = img.updateMask(crop_mask).reduceRegion(
                    reducer=ee.Reducer.mean(), geometry=roi, scale=1000, maxPixels=1e9
                ).getInfo()

                temp_k = stats.get('temperature_2m')
                temp_c = (temp_k - 273.15) if temp_k else np.nan
                sm_val = stats.get('volumetric_soil_water_layer_1')

                records.append({'Date': d1, 'Temperature_C': round(temp_c, 2) if temp_c else np.nan, 'Soil_Moisture': round(sm_val, 3) if sm_val else np.nan})
        except Exception:
            pass
        current += datetime.timedelta(days=1)

    return pd.DataFrame(records)


def fetch_rainfall_data(start_str, end_str):
    print("🌧️ Fetching GPM Rainfall Data (Sunamganj & Meghalaya)...")
    locs = {'Sunamganj_Center': [91.3992, 25.0714], 'Meghalaya_Border': [91.73, 25.27]}

    records = []
    current = datetime.datetime.strptime(start_str, '%Y-%m-%d')
    end = datetime.datetime.strptime(end_str, '%Y-%m-%d')

    while current <= end:
        d_str = current.strftime('%Y-%m-%d')
        d_ee = ee.Date(d_str)
        try:
            # FIX: V06 এর বদলে V07 এবং precipitationCal এর বদলে precipitation ব্যবহার করা হয়েছে
            rain_img = ee.ImageCollection("NASA/GPM_L3/IMERG_V07").filterDate(d_ee, d_ee.advance(1, 'day')).select('precipitation').mean()
            row = {'Date': d_str}
            for name, coords in locs.items():
                point = ee.Geometry.Point(coords).buffer(5000)
                stats = rain_img.reduceRegion(reducer=ee.Reducer.mean(), geometry=point, scale=10000).getInfo()
                val = stats.get('precipitation', 0)
                row[f'Rain_{name}'] = (val if val else 0) * 24 # Convert mm/hr to daily
            records.append(row)
        except Exception:
            pass
        current += datetime.timedelta(days=1)

    return pd.DataFrame(records)


def fetch_satellite_data(start_str, end_str):
    print("🛰️ Fetching Optical (NDVI) and Radar (VV) Data (5-day steps)...")
    roi = ee.FeatureCollection("FAO/GAUL/2015/level2").filter(ee.Filter.eq('ADM2_NAME', 'Sunamganj')).geometry()
    crop_mask = ee.Image("ESA/WorldCover/v100/2020").select('Map').eq(40).clip(roi)

    # FIX: copyProperties ব্যবহার করে তারিখ সংরক্ষণ করা হয়েছে
    def process_s2(img):
        return img.select(['B8', 'B4'], ['NIR', 'Red']).divide(10000).normalizedDifference(['NIR', 'Red']).rename('NDVI').copyProperties(img, ['system:time_start'])

    def process_l8(img):
        return img.select(['SR_B5', 'SR_B4'], ['NIR', 'Red']).multiply(0.0000275).add(-0.2).normalizedDifference(['NIR', 'Red']).rename('NDVI').copyProperties(img, ['system:time_start'])

    optical_col = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(roi).map(process_s2).merge(
                  ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").filterBounds(roi).map(process_l8))

    s1 = ee.ImageCollection("COPERNICUS/S1_GRD").filterBounds(roi).filter(ee.Filter.eq('instrumentMode', 'IW')).filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')).select('VV')

    records = []
    current = datetime.datetime.strptime(start_str, '%Y-%m-%d')
    end = datetime.datetime.strptime(end_str, '%Y-%m-%d')

    while current <= end:
        d1 = current.strftime('%Y-%m-%d')
        d2 = (current + datetime.timedelta(days=20)).strftime('%Y-%m-%d') # 20-day window

        opt_val, sar_val = np.nan, np.nan
        try:
            # Optical
            opt_window = optical_col.filterDate(d1, d2)
            if opt_window.size().getInfo() > 0:
                stats = opt_window.qualityMosaic('NDVI').updateMask(crop_mask).clip(roi).reduceRegion(reducer=ee.Reducer.mean(), geometry=roi, scale=30, maxPixels=1e10).getInfo()
                opt_val = stats.get('NDVI', np.nan)

            # Radar
            sar_window = s1.filterDate(d1, d2)
            if sar_window.size().getInfo() > 0:
                sar_stats = sar_window.median().updateMask(crop_mask).clip(roi).reduceRegion(reducer=ee.Reducer.mean(), geometry=roi, scale=30, maxPixels=1e10).getInfo()
                sar_val = sar_stats.get('VV', np.nan)

            records.append({'Date': d1, 'Mean_NDVI': opt_val, 'Radar_VV': sar_val})
        except Exception:
            pass

        current += datetime.timedelta(days=5) # 5 day steps

    return pd.DataFrame(records)


def fetch_dahiti_water_level(start_str, end_str):
    print("🌊 Fetching DAHITI API Water Level Data...")
    url = "https://dahiti.dgfi.tum.de/api/v2/download-water-level/"
    args = {
        'api_key': '362D57A0E203292F23FFE4524D8FC91632BF76AF9C7D9EE91C68EF7FF7F73CAA',
        'dahiti_id': 11199,
        'format': 'json',
        'action': 'download-water-level'
    }

    response = requests.post(url, json=args)
    if response.status_code == 200:
        data = response.json()
        water_data = data if isinstance(data, list) else data.get('water_level', [])

        if water_data:
            df = pd.DataFrame(water_data)
            date_col = 'date' if 'date' in df.columns else 'datetime'
            # Convert to pure date string (YYYY-MM-DD)
            df['Date'] = pd.to_datetime(df[date_col]).dt.strftime('%Y-%m-%d')
            df.rename(columns={'water_level': 'DAHITI_Water_Level'}, inplace=True)

            # Filter by target date range
            df = df[(df['Date'] >= start_str) & (df['Date'] <= end_str)]
            return df[['Date', 'DAHITI_Water_Level']].groupby('Date').mean().reset_index()

    print("⚠️ DAHITI Data not found or API error.")
    return pd.DataFrame(columns=['Date', 'DAHITI_Water_Level'])

# ==========================================
# 3. Execution & Merging Pipeline
# ==========================================

def build_master_dataset():
    print(f"🚀 Starting AgroPulse Data Pipeline ({START_DATE} to {END_DATE})")

    # 1. Create a Master Calendar
    date_range = pd.date_range(start=START_DATE, end=END_DATE)
    master_df = pd.DataFrame({'Date': date_range.strftime('%Y-%m-%d')})

    # 2. Fetch all components
    df_climate = fetch_climate_data(START_DATE, END_DATE)
    df_rain = fetch_rainfall_data(START_DATE, END_DATE)
    df_sat = fetch_satellite_data(START_DATE, END_DATE)
    df_water = fetch_dahiti_water_level(START_DATE, END_DATE)

    print("\n🔄 Merging all datasets...")
    # 3. Merge them to the Master Calendar
    for df in [df_climate, df_rain, df_sat, df_water]:
        if not df.empty:
            master_df = pd.merge(master_df, df, on='Date', how='left')

    # 4. Handle Missing Data (Interpolation)
    print("⚙️ Applying Interpolation for missing days...")

    master_df = master_df.sort_values('Date')
    master_df.set_index('Date', inplace=True)

    # Fill gaps using linear interpolation
    master_df = master_df.interpolate(method='linear', limit_direction='both')

    # If there are still NaNs at the very edges, forward/backward fill
    master_df = master_df.ffill().bfill()

    master_df.reset_index(inplace=True)

    # 5. Save to CSV
    final_csv_name = f'AgroPulse_Master_Dataset_{START_DATE}_to_{END_DATE}.csv'
    master_df.to_csv(final_csv_name, index=False)

    print(f"🎉 SUCCESS! Unified Dataset saved as: {final_csv_name}")
    print(master_df.head())

if __name__ == "__main__":
    build_master_dataset()

🚀 Starting AgroPulse Data Pipeline (2026-01-01 to 2026-03-18)
🌡️ Fetching ERA5 Climate Data (Temp & Soil Moisture)...
🌧️ Fetching GPM Rainfall Data (Sunamganj & Meghalaya)...
🛰️ Fetching Optical (NDVI) and Radar (VV) Data (5-day steps)...
🌊 Fetching DAHITI API Water Level Data...
⚠️ DAHITI Data not found or API error.

🔄 Merging all datasets...
⚙️ Applying Interpolation for missing days...
🎉 SUCCESS! Unified Dataset saved as: AgroPulse_Master_Dataset_2026-01-01_to_2026-03-18.csv
         Date  Temperature_C  Soil_Moisture  Rain_Sunamganj_Center  \
0  2026-01-01          16.90          0.163               0.000000   
1  2026-01-02          17.53          0.163               0.000274   
2  2026-01-03          16.52          0.162               0.000000   
3  2026-01-04          16.04          0.162               0.000886   
4  2026-01-05          15.76          0.162               0.000000   

   Rain_Meghalaya_Border  Mean_NDVI   Radar_VV  
0                    0.0   0.391906 -12.150190

In [ ]:
import pandas as pd
import io

# সোর্স ফাইলের নাম
SOURCE_CSV = 'AgroPulse_Master_Dataset_2026-01-01_to_2026-03-18.csv'
# নতুন সেভ করা ফাইলের নাম
OUTPUT_CSV = 'AgroPulse_Test_Dataset_Final.csv'

# ১. আপনার দেওয়া FFWC এর কাঁচা ডেটা (Raw Data)
manual_data = """Date_Time,Water_Level
19-03-2026 15:00:00,3.03
19-03-2026 12:00:00,3.04
19-03-2026 09:00:00,3.06
19-03-2026 06:00:00,3.09
18-03-2026 18:00:00,3.02
18-03-2026 15:00:00,3.05
18-03-2026 12:00:00,3.07
18-03-2026 09:00:00,3.01
18-03-2026 06:00:00,2.92
17-03-2026 18:00:00,2.70
17-03-2026 15:00:00,2.58
17-03-2026 12:00:00,2.44
16-03-2026 18:00:00,1.80
16-03-2026 15:00:00,1.69
16-03-2026 12:00:00,1.54
16-03-2026 09:00:00,1.40
16-03-2026 06:00:00,1.34
15-03-2026 18:00:00,1.00
15-03-2026 15:00:00,1.07
15-03-2026 12:00:00,1.03
15-03-2026 09:00:00,0.99
15-03-2026 06:00:00,0.96
14-03-2026 18:00:00,0.91
14-03-2026 15:00:00,0.87
14-03-2026 12:00:00,0.79
13-03-2026 18:00:00,0.80
13-03-2026 15:00:00,0.77
13-03-2026 12:00:00,0.72
13-03-2026 09:00:00,0.57
13-03-2026 06:00:00,0.59
12-03-2026 18:00:00,0.69
12-03-2026 15:00:00,0.67
12-03-2026 12:00:00,0.67
12-03-2026 09:00:00,0.67
12-03-2026 06:00:00,0.70
11-03-2026 18:00:00,0.77
11-03-2026 15:00:00,0.74
11-03-2026 12:00:00,0.75
11-03-2026 09:00:00,0.67
11-03-2026 06:00:00,0.70
10-03-2026 18:00:00,0.77
10-03-2026 15:00:00,0.84
10-03-2026 12:00:00,0.87
10-03-2026 09:00:00,0.90
10-03-2026 06:00:00,0.93
09-03-2026 18:00:00,0.96"""

print("🌊 Processing Real Water Level Data and Saving to New File...")

# ২. ডেটা রিড করে প্রতিদিনের গড় (Mean) বের করা
manual_df = pd.read_csv(io.StringIO(manual_data))
manual_df['Date'] = pd.to_datetime(manual_df['Date_Time'], format='%d-%m-%Y %H:%M:%S').dt.strftime('%Y-%m-%d')
daily_manual = manual_df.groupby('Date')['Water_Level'].mean().reset_index()

# ৩. ৩১ ডিসেম্বরের ডেটা যুক্ত করা (ইন্টারপোলেশনের স্টার্ট পয়েন্ট হিসেবে)
extra_row = pd.DataFrame({'Date': ['2025-12-31'], 'Water_Level': [6.40]})
daily_manual = pd.concat([extra_row, daily_manual], ignore_index=True)

# ৪. মেইন ডেটাসেট রিড করা
df = pd.read_csv(SOURCE_CSV)

# যদি আগে কোনো ডামি কলাম থাকে, তা সরিয়ে দেওয়া
cols_to_drop = ['DAHITI_Water_Level', 'Daily_Water_Level']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# ৫. ৩১ ডিসেম্বর থেকে ডেটাসেটের শেষ দিন পর্যন্ত ক্যালেন্ডার তৈরি ও ইন্টারপোলেশন
temp_dates = pd.date_range(start='2025-12-31', end=df['Date'].max()).strftime('%Y-%m-%d')
temp_df = pd.DataFrame({'Date': temp_dates})
temp_df = pd.merge(temp_df, daily_manual, on='Date', how='left')

print("⚙️ Interpolating missing days between Dec 31 (6.40m) and Mar 09 (0.96m)...")
temp_df['Daily_Water_Level'] = temp_df['Water_Level'].interpolate(method='linear')

# ৬. নতুন কলামটি মেইন ডেটাসেটে যুক্ত করা
df = pd.merge(df, temp_df[['Date', 'Daily_Water_Level']], on='Date', how='left')
df['Daily_Water_Level'] = df['Daily_Water_Level'].round(2)

# ৭. নতুন CSV ফাইলে সেভ করা
df.to_csv(OUTPUT_CSV, index=False)

print(f"🎉 Success! New dataset saved as: {OUTPUT_CSV}")
print("\n--- Top Rows ---")
print(df.head())

🌊 Processing Real Water Level Data and Saving to New File...
⚙️ Interpolating missing days between Dec 31 (6.40m) and Mar 09 (0.96m)...
🎉 Success! New dataset saved as: AgroPulse_Test_Dataset_Final.csv

--- Top Rows ---
         Date  Temperature_C  Soil_Moisture  Rain_Sunamganj_Center  \
0  2026-01-01          16.90          0.163               0.000000   
1  2026-01-02          17.53          0.163               0.000274   
2  2026-01-03          16.52          0.162               0.000000   
3  2026-01-04          16.04          0.162               0.000886   
4  2026-01-05          15.76          0.162               0.000000   

   Rain_Meghalaya_Border  Mean_NDVI   Radar_VV  Daily_Water_Level  
0                    0.0   0.391906 -12.150190               6.32  
1                    0.0   0.399414 -12.074448               6.24  
2                    0.0   0.406923 -11.998705               6.16  
3                    0.0   0.414432 -11.922962               6.08  
4                  